[Markdown Cell]
# Model D — faster full-corpus training

Attach your **complete tokenized dataset** and optionally the corresponding saved
original/auto model output. Select **T4 x2** and Internet, then Save & Run All.
No corpus sampling, fewer epochs, shorter training windows, or smaller model.
Fresh runs retain five epochs; parent continuation retains two at LR $10^{-4}$.
Effective global batch, optimizer schedule, dropout and precision are preserved.

The optimized runtime uses fused CUDA AdamW, one prefetched data worker per rank,
CPU-prepared supervision indices, bounded 512-token vocabulary loss chunks,
implicit SDPA masks for unpadded batches, and one shared finite check per update.
A/C/D cache FP32 rotary tables; D also caches exact sparse training routes.
Every real window appears once per epoch: ignored sentinels synchronize the tail.
No repeated real example pads a batch. Model C retains FP32 GRU arithmetic and
checkpoints Transformer blocks while retaining recurrent activations, avoiding
costly GRU recomputation. Its optional recurrent_checkpointing setting restores
stage recomputation when a custom architecture needs the lower memory footprint.

Checkpoints still include every training state and both rank RNGs every 500 updates,
every epoch and graceful stop. Fast-stage checkpoints resume their exact cursor.
An older original/auto checkpoint initializes a **new** fast stage with fresh
optimizer/schedule; its former cursor is not claimed as an exact fast-stage resume.
Use its original notebook if you need to finish that older stage exactly.

Outputs: `/kaggle/working/model_d_fast_run/`. Preserve the full saved output.
Use the bundled generalized evaluator with that output, tokenized export and test.json.
Measured tokens/second and seconds/update appear in rank-0 progress and JSONL logs.
Full dual-T4 throughput is hardware-dependent and must be measured on Kaggle.


[Markdown Cell]
## 1. Environment and imports
Use Kaggle's CUDA-enabled PyTorch >=2.5. Only small tokenizer/progress packages are installed. T4 uses FP16 autocast with dynamic loss scaling; PyTorch chooses a supported attention kernel.


In [1]:
# [Code Cell]
%pip install -q "tokenizers==0.22.1" "tqdm==4.67.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# [Code Cell]
from pathlib import Path
from dataclasses import asdict, replace
import importlib
import json
import os
import signal
import sys
import subprocess
import torch
import numpy as np
from tokenizers import Tokenizer
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('Visible GPUs:', torch.cuda.device_count())
module_parts: dict[str, list[str]] = {}

def append_module(filename: str, source: str) -> None:
    """Keep a readable runtime section until the importable package is assembled."""
    assert filename.endswith('.py') and '/' not in filename and ('\\' not in filename)
    module_parts.setdefault(filename, []).append(source)


Python: 3.13.15 PyTorch: 2.11.0+cu128
Visible GPUs: 2


[Markdown Cell]
## 2. Configuration and strict parameter budget
For vocabulary $V$, width $D$, FFN width $F$, and $L$ sparse decoder layers,
\[
P=VD+L(4D^2+3DF+2D)+D=110{,}119{,}680.
\]
Routing indices and RoPE add no learned weights; the LM head shares the embedding.
Count unique trainable Parameters and assert $50{,}000{,}000<P<300{,}000{,}000$.
Defaults: 12 layers, 12 heads, width 768, SwiGLU width 2048, local window 128,
16 dilated keys, landmark stride 256, and 32 queries per attention chunk.
Layer dilation cycles through 64, 128, 256, 512. Dropout is 0.1.

This fast stage trains on up to 1536 BPE positions, one window per GPU, with 8 microbatches per update:
at most 24,576 supervised tokens globally. Capacity is 2048 positions; reserve
1536 prompt and 512 target positions for validation. Words and BPE tokens differ;
report all crops. Change architecture/settings in a fresh run directory.


[Markdown Cell]
config.py: imports and module constants


In [3]:
# [Code Cell]
append_module('config.py', r'''from dataclasses import asdict, dataclass, replace
from pathlib import Path
import json

RUNTIME_VERSION = "cs781-model-d-v1"
SPECIAL_TOKENS = ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")
SOURCES = ("supreme_court", "high_courts", "acts", "tribunals")


''')


[Markdown Cell]
config.py: TrainConfig


In [4]:
# [Code Cell]
append_module('config.py', r'''@dataclass(frozen=True)
class TrainConfig:
    """Immutable architecture, causal objective, and training/recovery choices."""

    data_dir: str = "/kaggle/input/cs781-preprocessed-data"
    run_dir: str = "/kaggle/working/model_d_run"
    resume_dir: str | None = None
    vocab_size: int = 32_768
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 12
    d_ff: int = 2_048
    rms_epsilon: float = 1e-6
    rope_base: float = 10_000.0
    max_seq_tokens: int = 2_048
    train_seq_tokens: int = 1_024
    max_prompt_tokens: int = 1_536  # Includes BOS; words and BPE tokens differ.
    max_target_tokens: int = 512   # Includes EOS if the reference fits.
    dropout: float = 0.1
    activation_checkpointing: bool = True
    loss_chunk_tokens: int = 128
    local_window: int = 128  # Includes the query's own key.
    dilation_base: int = 64
    dilation_cycle: int = 4  # Layer strides: 64, 128, 256, 512, then repeat.
    dilated_links: int = 16
    global_stride: int = 256  # Absolute landmark keys: 0, 256, 512, ...
    attention_query_chunk: int = 32
    sparse_attention_checkpointing: bool = True
    epochs: int = 1
    micro_batch_size: int = 2  # Per T4; effective global batch = 16 windows.
    accumulation_steps: int = 4
    world_size: int = 2
    num_workers: int = 0
    learning_rate: float = 3e-4
    min_lr_ratio: float = 0.1
    warmup_updates: int = 500
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    max_run_seconds: int = 37_800
    min_free_bytes: int = 1 << 30
    max_checkpoint_bytes: int = 6 << 30
    seed: int = 781
    deterministic: bool = False
    generation_examples_per_source: int = 2
    generation_token_limit: int = 512
    experiment_kind: str = "cs781-fast-stage-v1"
    initial_checkpoint: str | None = None
    initial_checkpoint_sha256: str | None = None
    parent_data_fingerprint: str | None = None
    optimization_version: str = "full-corpus-fast-v1"
    fused_adamw: bool = True
    loader_prefetch_factor: int = 2

''')


[Markdown Cell]
config.py: TrainConfig.block_tokens


In [5]:
# [Code Cell]
append_module('config.py', r'''    @property
    def block_tokens(self) -> int:
        """Window stride: reuse one context token so document transitions survive."""
        return self.train_seq_tokens

''')


[Markdown Cell]
config.py: TrainConfig.validate


In [6]:
# [Code Cell]
append_module('config.py', r'''    def validate(self) -> None:
        assert self.vocab_size == 32_768
        assert self.d_model > 0 and self.n_heads > 0 and self.d_model % self.n_heads == 0
        assert (self.d_model // self.n_heads) % 2 == 0, "RoPE needs even head width"
        assert self.n_layers > 0 and self.d_ff > 0 and self.rms_epsilon > 0 and self.rope_base > 1
        assert 0 <= self.dropout < 1
        assert 1 <= self.local_window <= self.max_seq_tokens
        assert self.dilation_base >= 1 and 1 <= self.dilation_cycle <= 8
        assert self.dilated_links >= 1 and 1 <= self.global_stride <= self.max_seq_tokens
        assert 1 <= self.attention_query_chunk <= self.max_seq_tokens
        assert 1 <= self.train_seq_tokens <= self.max_seq_tokens
        assert self.max_prompt_tokens >= 2 and self.max_target_tokens >= 1
        assert self.max_prompt_tokens + self.max_target_tokens - 1 <= self.max_seq_tokens
        assert 1 <= self.loss_chunk_tokens <= self.max_seq_tokens
        assert self.epochs > 0 and self.micro_batch_size > 0 and self.accumulation_steps > 0
        assert self.loader_prefetch_factor > 0 and self.optimization_version == "full-corpus-fast-v1"
        assert isinstance(self.fused_adamw, bool)
        assert self.world_size == 2 and self.num_workers >= 0
        assert self.checkpoint_every_steps == 500
        assert self.learning_rate > 0 and 0 < self.min_lr_ratio <= 1
        assert self.warmup_updates >= 0 and self.max_grad_norm > 0
        assert 0 <= self.weight_decay and 0 < self.beta1 < 1 and 0 < self.beta2 < 1
        assert self.adam_epsilon > 0 and self.max_consecutive_overflows > 0
        assert self.max_run_seconds > 0 and self.min_free_bytes >= 0 and self.max_checkpoint_bytes > 0
        assert self.generation_examples_per_source > 0
        assert 1 <= self.generation_token_limit <= self.max_seq_tokens - self.max_prompt_tokens + 1

''')


[Markdown Cell]
config.py: TrainConfig.load


In [7]:
# [Code Cell]
append_module('config.py', r'''    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        return cls(**json.loads(Path(path).read_text(encoding="utf-8")))

''')


[Markdown Cell]
config.py: TrainConfig.save


In [8]:
# [Code Cell]
append_module('config.py', r'''    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2) + "\n", encoding="utf-8")


''')


[Markdown Cell]
config.py: unique_trainable_parameters


In [9]:
# [Code Cell]
append_module('config.py', r'''def unique_trainable_parameters(model) -> int:
    """Count each shared Parameter once; the tied LM head adds no new weights."""
    return sum(p.numel() for p in {id(p): p for p in model.parameters() if p.requires_grad}.values())


''')


[Markdown Cell]
config.py: assert_parameter_boundary


In [10]:
# [Code Cell]
append_module('config.py', r'''def assert_parameter_boundary(model) -> int:
    count = unique_trainable_parameters(model)
    assert 50_000_000 < count < 300_000_000, f"Parameter boundary violated: {count:,}"
    return count

REQUIRED_DATA_FILES = (
    "metadata.json", "tokenizer.json", "train.bin", "val.bin",
    "train_documents.jsonl", "validation_documents.jsonl", "validation_pairs.jsonl",
)


''')


[Markdown Cell]
config.py: complete_preprocessing_directory


In [11]:
# [Code Cell]
append_module('config.py', r'''def complete_preprocessing_directory(directory: Path) -> bool:
    """Check names and receipt sizes without loading binaries or copying input data.

    ZIP exports may contain a second, incomplete downloaded_output directory.
    Empty binaries and incomplete exports must never win automatic discovery.
    Full SHA-256 verification still runs in the existing data-contract check.
    """
    try:
        if not all((directory / name).is_file() for name in REQUIRED_DATA_FILES):
            return False
        metadata_path = directory / "metadata.json"
        if not 0 < metadata_path.stat().st_size <= (4 << 20):
            return False
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if not isinstance(metadata, dict) or metadata.get("status") != "verified":
            return False
        if metadata.get("schema_version") != 1 or metadata.get("dtype") != "<u2":
            return False
        if metadata.get("vocab_size") != 32_768:
            return False
        for name, key in (("train.bin", "train_tokens"), ("val.bin", "val_tokens")):
            count = metadata[key]
            if type(count) is not int or count <= 0 or (directory / name).stat().st_size != count * 2:
                return False
        for name in REQUIRED_DATA_FILES[1:]:
            size = (directory / name).stat().st_size
            if size <= 0 or size != metadata["artifacts"][name]["bytes"]:
                return False
        return True
    except (OSError, ValueError, KeyError, TypeError):
        return False


''')


[Markdown Cell]
config.py: resolve_preprocessed_data_dir


In [12]:
# [Code Cell]
append_module('config.py', r'''def resolve_preprocessed_data_dir(requested: str | Path,
                                  input_root: str | Path = "/kaggle/input") -> Path:
    """Prefer a complete explicit folder, otherwise locate one unique mounted export.

    Multiple complete exports require an explicit choice; files are never copied
    from read-only Kaggle input into scarce working storage.
    """
    requested_path = Path(requested).expanduser()
    if complete_preprocessing_directory(requested_path):
        return requested_path.resolve()
    roots = [requested_path] if requested_path.is_dir() else []
    mounted_root = Path(input_root)
    if mounted_root.is_dir() and mounted_root not in roots:
        roots.append(mounted_root)
    candidates: set[Path] = set()
    inspected: set[Path] = set()
    for root in roots:
        for metadata_path in root.rglob("metadata.json"):
            directory = metadata_path.parent.resolve()
            inspected.add(directory)
            if complete_preprocessing_directory(directory):
                candidates.add(directory)
    if len(candidates) == 1:
        return next(iter(candidates))
    if len(candidates) > 1:
        options = "\n".join(f"  {path}" for path in sorted(candidates))
        raise ValueError("Multiple complete preprocessing exports found. Set cfg.data_dir "
                         f"to one exact folder:\n{options}")
    found = "\n".join(f"  {path}" for path in sorted(inspected)[:8]) or "  none"
    raise FileNotFoundError(
        f"No complete preprocessing export at {requested_path} or below {mounted_root}. "
        "Attach the tokenized dataset and select the folder containing metadata.json, "
        "tokenizer.json, both nonempty .bin files, and the three JSONL manifests. "
        f"Inspected metadata folders (incomplete or invalid):\n{found}"
    )


''')


[Markdown Cell]
config.py: resolve_training_config


In [13]:
# [Code Cell]
append_module('config.py', r'''def resolve_training_config(cfg: TrainConfig,
                            input_root: str | Path = "/kaggle/input") -> TrainConfig:
    """Canonicalize the input path before saving config or spawning GPU workers."""
    resolved = resolve_preprocessed_data_dir(cfg.data_dir, input_root)
    print(f"Preprocessed data: {resolved}", flush=True)
    return replace(cfg, data_dir=str(resolved))

import ast
import hashlib

AUTO_STAGE = "cs781-fast-stage-v1"


''')


[Markdown Cell]
config.py: auto_sha256


In [14]:
# [Code Cell]
append_module('config.py', r'''def auto_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


''')


[Markdown Cell]
config.py: auto_saved_runs


In [15]:
# [Code Cell]
append_module('config.py', r'''def auto_saved_runs(input_root: Path, family: str) -> list[Path]:
    """Discover output folders; notebook execution logs never count as weights."""
    if not input_root.is_dir():
        return []
    return sorted({path.parent.resolve() for path in input_root.rglob("train_config.json")
                   if (path.parent / "code" / f"model_{family}" / "model.py").is_file()
                   and any((path.parent / "checkpoints" / name).is_file()
                           for name in ("best.pt", "slot_0.pt", "slot_1.pt"))})


''')


[Markdown Cell]
config.py: auto_parent_checkpoint


In [16]:
# [Code Cell]
append_module('config.py', r'''def auto_parent_checkpoint(run: Path) -> tuple[Path, dict]:
    """Prefer best validation weights; require the selected checkpoint's receipt."""
    choices = []
    for name in ("best.pt", "slot_0.pt", "slot_1.pt"):
        path = run / "checkpoints" / name
        if not path.is_file():
            continue
        receipt_path = path.with_name(path.name + ".json")
        if not receipt_path.is_file():
            raise FileNotFoundError(f"Checkpoint receipt missing: {receipt_path}")
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        if type(receipt.get("step")) is not int or receipt["step"] <= 0:
            raise ValueError(f"No successful saved progress in {path}")
        choices.append((path, receipt))
    if not choices:
        raise FileNotFoundError(f"No committed checkpoint in {run}/checkpoints")
    selected = next((item for item in choices if item[0].name == "best.pt"), None)
    path, receipt = selected or max(choices, key=lambda item: (item[1]["step"], item[1]["generation"]))
    if path.stat().st_size != receipt["bytes"] or auto_sha256(path) != receipt["sha256"]:
        raise ValueError(f"Selected parent checkpoint failed SHA-256/size verification: {path}")
    return path, receipt


''')


[Markdown Cell]
config.py: auto_data_fingerprint


In [17]:
# [Code Cell]
append_module('config.py', r'''def auto_data_fingerprint(data_dir: Path) -> str:
    """Derive the original identity; full artifact SHA verification follows in train.py."""
    metadata = json.loads((data_dir / "metadata.json").read_text(encoding="utf-8"))
    names = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",
             "validation_documents.jsonl", "validation_pairs.jsonl")
    identity = {name: metadata["artifacts"][name]["sha256"] for name in names}
    identity["metadata.json"] = auto_sha256(data_dir / "metadata.json")
    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()


''')


[Markdown Cell]
config.py: auto_configuration


In [18]:
# [Code Cell]
append_module('config.py', r'''def auto_configuration(family: str, model_ast_sha256: str, data_dir: str,
                       run_dir: str, parent_run: str | None = None,
                       input_root: str = "/kaggle/input") -> tuple[TrainConfig, dict]:
    """Warm-start old weights once; resume this stage without changing its schedule.

    A new continuation resets Adam, scaler, scheduler, epoch/step, and rank RNGs.
    An interrupted auto stage instead restores all of them through CheckpointManager.
    """
    data_path = resolve_preprocessed_data_dir(data_dir, input_root)
    working = Path(run_dir)
    local_config = working / "train_config.json"
    if local_config.is_file() and any((working / "checkpoints" / name).is_file()
                                      for name in ("best.pt", "slot_0.pt", "slot_1.pt")):
        saved = json.loads(local_config.read_text(encoding="utf-8"))
        if saved.get("experiment_kind") != AUTO_STAGE:
            raise ValueError("The working folder belongs to another experiment; choose a fresh output folder")
        cfg = replace(TrainConfig(**saved), data_dir=str(data_path), run_dir=str(working), resume_dir=None)
        return cfg, {"mode": "resume_working_stage", "saved_config": str(local_config)}
    runs = [Path(parent_run).resolve()] if parent_run else auto_saved_runs(Path(input_root), family)
    if parent_run and not (runs[0] / "train_config.json").is_file():
        raise FileNotFoundError(f"Select the run root, containing train_config.json: {runs[0]}")
    stage_runs = [run for run in runs if
                  json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == AUTO_STAGE]
    old_auto_runs = [run for run in runs if json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == "cs781-auto-stage-v1"]
    if not stage_runs and len(old_auto_runs) == 1:
        runs = old_auto_runs
    if len(stage_runs) > 1 or (not stage_runs and len(runs) > 1):
        raise ValueError(f"Several Model {family.upper()} runs match; set PARENT_RUN_DIR to one exact run root: {runs}")
    if stage_runs:
        run = stage_runs[0]
        saved = json.loads((run / "train_config.json").read_text(encoding="utf-8"))
        cfg = replace(TrainConfig(**saved), data_dir=str(data_path), run_dir=str(working),
                      resume_dir=str(run / "checkpoints"))
        return cfg, {"mode": "resume_attached_stage", "saved_run": str(run)}
    base = TrainConfig()
    # Prefer the latest auto stage over an original run if both are attached.
    auto_runs = [run for run in runs if json.loads((run / "train_config.json").read_text(encoding="utf-8")).get("experiment_kind") == "cs781-auto-stage-v1"]
    if not stage_runs and len(auto_runs) == 1:
        runs = auto_runs
    parent_path, parent_receipt, parent_identity = None, None, None
    if runs:
        run = runs[0]
        previous = json.loads((run / "train_config.json").read_text(encoding="utf-8"))
        base = TrainConfig(**previous)
        parent_source = (run / "code" / f"model_{family}" / "model.py").read_text(encoding="utf-8")
        parent_model_digest = hashlib.sha256(ast.dump(ast.parse(parent_source)).encode()).hexdigest()
        if parent_model_digest not in ({model_ast_sha256} if isinstance(model_ast_sha256, str) else set(model_ast_sha256)):
            raise ValueError("Parent architecture code differs; this adapter supports your original prepared A/B/C/D implementations")
        data_report = run / "data_report.json"
        if not data_report.is_file():
            data_report = run / "indices" / "report.json"
        if not data_report.is_file():
            raise FileNotFoundError("Attach the full saved run, including data_report.json; tokenizer/data compatibility must be proved")
        parent_identity = json.loads(data_report.read_text(encoding="utf-8")).get("data_fingerprint")
        if not parent_identity or parent_identity != auto_data_fingerprint(data_path):
            raise ValueError("Parent tokenizer/data export differs from the attached preprocessing dataset")
        parent_path, parent_receipt = auto_parent_checkpoint(run)
    settings = dict(
    data_dir=str(data_path), run_dir=str(working), resume_dir=None,
    experiment_kind=AUTO_STAGE, initial_checkpoint=str(parent_path) if parent_path else None,
    initial_checkpoint_sha256=parent_receipt["sha256"] if parent_receipt else None,
    parent_data_fingerprint=parent_identity,
    epochs=1, learning_rate=1e-4 if runs else 3e-4,
    warmup_updates=250 if runs else 500, 
    micro_batch_size=2,            # Changed from 1
    accumulation_steps=4,          # Changed from 8
    activation_checkpointing=False, # Changed from True
    num_workers=2,                 # Changed from 1
    optimization_version="full-corpus-fast-v1", fused_adamw=True, loader_prefetch_factor=2,
    loss_chunk_tokens=min(512, base.max_decoder_tokens if family == "b" else base.max_seq_tokens),
    generation_token_limit=min(512, base.max_decoder_tokens) if family == "b" else
        min(512, base.max_seq_tokens - base.max_prompt_tokens + 1)
)
    if family == "b":
        settings.update(train_prefix_tokens=min(1536, base.max_source_tokens - 1),
                        train_target_tokens=min(512, base.max_decoder_tokens))
    else:
        settings.update(train_seq_tokens=min(1536, base.max_seq_tokens))
    cfg = replace(base, **settings)
    cfg.validate()
    report = {"mode": "continue_parent_weights" if runs else "fresh_random_initialization",
              "parent_run": str(runs[0]) if runs else None,
              "parent_checkpoint": str(parent_path) if parent_path else None,
              "parent_sha256": parent_receipt["sha256"] if parent_receipt else None,
              "parent_epoch": parent_receipt["epoch"] if parent_receipt else None,
              "parent_step": parent_receipt["step"] if parent_receipt else None,
              "stage_epochs": cfg.epochs, "stage_learning_rate": cfg.learning_rate,
              "optimizer_schedule": "New Adam/scaler/cosine schedule; exact full-state resume after first stage checkpoint"}
    return cfg, report
''')


[Markdown Cell]
The preferred input is `/kaggle/input/cs781-preprocessed-data/`. Discover the unique complete export even inside `kaggle/working/downloaded_output/` ZIP folders under the mounted dataset. Those names describe the old export, not the live working path. Size receipts reject empty duplicate binaries; multiple complete exports require an explicit `data_dir`. Full SHA-256 verification follows.


[Markdown Cell]
Discover saved runs without running an uploaded notebook


[Markdown Cell]
Choose fresh training, new continuation, or exact stage resumption


In [19]:
# [Code Cell]
exec(compile("".join(module_parts["config.py"]), "model_d/config.py", "exec"), globals())
PARENT_RUN_DIR: str | None = None  # Leave None for automatic discovery.
cfg, auto_training_report = auto_configuration(
    "d", ('5654bf6c9b86080fdb35169dbf9903c8c1a6e85aebd3b0b4cc3dea0e5c167bea', 'd78016f6e93f7960cf87b260b9b46de0fce21c5c1816b60f47fb081408d5dfd0'), "/kaggle/input/cs781-preprocessed-data",
    "/kaggle/working/model_d_fast_run", parent_run=PARENT_RUN_DIR,
)
cfg.validate()
OFFICIAL_TEST_JSON: str | None = None
Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
(Path(cfg.run_dir) / "auto_training_report.json").write_text(
    json.dumps(auto_training_report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(auto_training_report, indent=2))
print(json.dumps(asdict(cfg), indent=2))


{
  "mode": "fresh_random_initialization",
  "parent_run": null,
  "parent_checkpoint": null,
  "parent_sha256": null,
  "parent_epoch": null,
  "parent_step": null,
  "stage_epochs": 1,
  "stage_learning_rate": 0.0003,
  "optimizer_schedule": "New Adam/scaler/cosine schedule; exact full-state resume after first stage checkpoint"
}
{
  "data_dir": "/kaggle/input/datasets/azadbharti/dataset-tokenized-smallest-25m",
  "run_dir": "/kaggle/working/model_d_fast_run",
  "resume_dir": null,
  "vocab_size": 32768,
  "d_model": 768,
  "n_heads": 12,
  "n_layers": 12,
  "d_ff": 2048,
  "rms_epsilon": 1e-06,
  "rope_base": 10000.0,
  "max_seq_tokens": 2048,
  "train_seq_tokens": 1536,
  "max_prompt_tokens": 1536,
  "max_target_tokens": 512,
  "dropout": 0.1,
  "activation_checkpointing": false,
  "loss_chunk_tokens": 512,
  "local_window": 128,
  "dilation_base": 64,
  "dilation_cycle": 4,
  "dilated_links": 16,
  "global_stride": 256,
  "attention_query_chunk": 32,
  "sparse_attention_checkpoint

[Markdown Cell]
## 3. Normalization and rotary positions
\[
\operatorname{RMSNorm}(h)=g\odot h/\sqrt{D^{-1}\sum_i h_i^2+\epsilon}.
\]
Normalize in FP32 before casting back. For adjacent Q/K components and
$\theta_i=10000^{-2i/d_h}$, position $m$ applies
\[
R_m(a,b)=(a\cos(m\theta_i)-b\sin(m\theta_i),\,
a\sin(m\theta_i)+b\cos(m\theta_i)).
\]
The Q/K dot product depends on relative positions. Trigonometry uses FP32.


[Markdown Cell]
model.py: imports and module constants


In [20]:
# [Code Cell]
append_module('model.py', r'''from dataclasses import dataclass
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from .config import TrainConfig, assert_parameter_boundary

KeyValue = tuple[Tensor, Tensor]


''')


[Markdown Cell]
model.py: RMSNorm


In [21]:
# [Code Cell]
append_module('model.py', r'''class RMSNorm(nn.Module):
    """RMSNorm: hidden[B,T,D] -> normalized[B,T,D]; FP32 variance reduction."""

    def __init__(self, width: int, epsilon: float) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.epsilon = epsilon

    def forward(self, hidden: Tensor) -> Tensor:
        normalized = hidden.float() * torch.rsqrt(hidden.float().square().mean(-1, keepdim=True) + self.epsilon)
        return (normalized * self.weight.float()).to(hidden.dtype)


''')


[Markdown Cell]
model.py: RotaryEmbedding


In [22]:
# [Code Cell]
append_module('model.py', r'''class RotaryEmbedding(nn.Module):
    """Rotate adjacent Q/K pairs at absolute positions; no learned position table.

    q,k[B,H,T,Dh], positions[T] -> rotated q,k of the same shape.
    Frequencies/trigonometry stay FP32 even under FP16 autocast.
    """

    def __init__(self, head_dim: int, base: float, max_positions: int = 2048) -> None:
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim, self.base, self.max_positions = head_dim, base, max_positions
        frequencies = base ** (-torch.arange(0, head_dim, 2, dtype=torch.float32, device="cpu") / head_dim)
        angles = torch.arange(max_positions, dtype=torch.float32, device="cpu")[:, None] * frequencies[None]
        self.register_buffer("cosine_cache", angles.cos(), persistent=False)
        self.register_buffer("sine_cache", angles.sin(), persistent=False)

    def forward(self, query: Tensor, key: Tensor, positions: Tensor) -> KeyValue:
        """Rotate [B,H,T,Dh] using cached FP32 trigonometry at true positions[T]."""
        with torch.autocast(device_type=query.device.type, enabled=False):
            cos_cache = self.cosine_cache.to(positions.device)
            sin_cache = self.sine_cache.to(positions.device)
            cosine = cos_cache.index_select(0, positions)[None, None]
            sine = sin_cache.index_select(0, positions)[None, None]
            def rotate(value: Tensor) -> Tensor:
                even, odd = value.float()[..., 0::2], value.float()[..., 1::2]
                return torch.stack((even * cosine - odd * sine, even * sine + odd * cosine),
                                   dim=-1).flatten(-2).to(value.dtype)
            return rotate(query), rotate(key)


''')

[Markdown Cell]
model.py: sparse_key_indices


In [23]:
# [Code Cell]
append_module('model.py', r'''def sparse_key_indices(positions: Tensor, key_length: int, local_window: int,
                       dilation: int, dilated_links: int, global_stride: int) -> tuple[Tensor, Tensor]:
    """positions[Q] -> indices[Q,K], allowed[Q,K] without allocating a T*T mask.

    Local keys have distance 0..W-1. Dilated keys have distance W+j*d,
    j=0..J-1. Global landmark keys have absolute index divisible by G.
    Every key must be past/present. Sort and mask repeated indices so overlapping
    routes never count a key multiple times in the softmax. Invalid entries are
    safely clamped for gathering but always excluded by the returned mask.
    """
    assert positions.ndim == 1 and positions.dtype == torch.long and key_length > 0
    device = positions.device
    local = positions[:, None] - torch.arange(local_window, device=device)[None]
    distances = local_window + torch.arange(dilated_links, device=device) * dilation
    dilated = positions[:, None] - distances[None]
    landmarks = torch.arange(0, key_length, global_stride, device=device)
    global_keys = landmarks[None].expand(len(positions), -1)
    candidates = torch.cat((local, dilated, global_keys), dim=1)
    legal = (candidates >= 0) & (candidates <= positions[:, None]) & (candidates < key_length)
    ordered = candidates.masked_fill(~legal, key_length).sort(dim=-1).values
    unique = torch.ones_like(ordered, dtype=torch.bool)
    unique[:, 1:] = ordered[:, 1:] != ordered[:, :-1]
    allowed = unique & (ordered < key_length)
    return ordered.clamp(0, key_length - 1), allowed


''')


[Markdown Cell]
model.py: SparseCausalSelfAttention


In [24]:
# [Code Cell]
append_module('model.py', r'''class SparseCausalSelfAttention(nn.Module):
    """hidden[B,T,D] -> hidden[B,T,D] with true gathered sparse key/value reads.

    Layer l uses dilation base*2**(l % cycle). SDPA sees Qc independent queries
    with at most W+J+ceil(T/G) candidate keys; there is no dense T*T score tensor.
    Attention-chunk checkpointing releases gathered K/V between forward and
    backward. The same attend() routing serves training, prefill, and decode.
    """

''')


[Markdown Cell]
model.py: SparseCausalSelfAttention.__init__


In [25]:
# [Code Cell]
append_module('model.py', r'''    def __init__(self, cfg: TrainConfig, layer_index: int) -> None:
        super().__init__()
        self.heads, self.head_dim = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.dropout = cfg.dropout
        self.local_window, self.dilated_links, self.global_stride = (
            cfg.local_window, cfg.dilated_links, cfg.global_stride)
        self.dilation = cfg.dilation_base * 2 ** (layer_index % cfg.dilation_cycle)
        self.query_chunk = cfg.attention_query_chunk
        self.checkpoint_chunks = cfg.sparse_attention_checkpointing
        self._route_cache_key, self._route_cache = None, None
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rotary = RotaryEmbedding(self.head_dim, cfg.rope_base, cfg.max_seq_tokens)

''')


[Markdown Cell]
model.py: SparseCausalSelfAttention.project


In [26]:
# [Code Cell]
append_module('model.py', r'''    def project(self, hidden: Tensor, positions: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        batch, length, _ = hidden.shape
        query, key, value = self.qkv(hidden).view(batch, length, 3, self.heads, self.head_dim).unbind(2)
        query, key, value = (tensor.transpose(1, 2) for tensor in (query, key, value))
        query, key = self.rotary(query, key, positions)
        return query, key, value

''')


[Markdown Cell]
model.py: SparseCausalSelfAttention.attend_chunk


In [27]:
# [Code Cell]
append_module('model.py', r'''    def attend_chunk(self, query: Tensor, key: Tensor, value: Tensor, positions: Tensor,
                     key_valid: Tensor | None) -> Tensor:
        """q[B,H,Qc,Dh], k/v[B,H,T,Dh] -> attended[B,H,Qc,Dh]."""
        if self.training and torch.is_grad_enabled():
            cache_key = (key.device, key.shape[2])
            if self._route_cache_key != cache_key:
                all_positions = torch.arange(key.shape[2], device=key.device)
                self._route_cache = sparse_key_indices(
                    all_positions, key.shape[2], self.local_window,
                    self.dilation, self.dilated_links, self.global_stride)
                self._route_cache_key = cache_key
            indices, allowed = (part.index_select(0, positions) for part in self._route_cache)
        else:
            # One-token generation computes only its own routes.
            indices, allowed = sparse_key_indices(
                positions, key.shape[2], self.local_window,
                self.dilation, self.dilated_links, self.global_stride)
        batch, heads, queries, width = query.shape
        candidates = indices.shape[1]
        # Gather only selected keys; query chunks bound this materialized tensor.
        selected_key = key[:, :, indices, :].permute(0, 2, 1, 3, 4).reshape(
            batch * queries, heads, candidates, width)
        selected_value = value[:, :, indices, :].permute(0, 2, 1, 3, 4).reshape(
            batch * queries, heads, candidates, width)
        valid = allowed[None].expand(batch, -1, -1)
        if key_valid is not None:
            valid = valid & key_valid[:, indices]
        mask = valid.reshape(batch * queries, 1, 1, candidates)
        q = query.permute(0, 2, 1, 3).reshape(batch * queries, heads, 1, width)
        attended = F.scaled_dot_product_attention(
            q, selected_key, selected_value, attn_mask=mask, is_causal=False,
            dropout_p=self.dropout if self.training else 0.0)
        return attended.reshape(batch, queries, heads, width).permute(0, 2, 1, 3)

''')


[Markdown Cell]
model.py: SparseCausalSelfAttention.attend


In [28]:
# [Code Cell]
append_module('model.py', r'''    def attend(self, query: Tensor, kv: KeyValue, positions: Tensor,
               key_valid: Tensor | None = None) -> Tensor:
        """Rotated queries at absolute positions attend to the full bounded K/V cache."""
        pieces = []
        for start in range(0, len(positions), self.query_chunk):
            q = query[:, :, start:start + self.query_chunk]
            pos = positions[start:start + self.query_chunk]
            if self.training and torch.is_grad_enabled() and self.checkpoint_chunks:
                pieces.append(checkpoint(self.attend_chunk, q, kv[0], kv[1], pos, key_valid,
                                         use_reentrant=False))
            else:
                pieces.append(self.attend_chunk(q, kv[0], kv[1], pos, key_valid))
        output = torch.cat(pieces, dim=2)
        return self.out(output.transpose(1, 2).contiguous().flatten(2))

''')


[Markdown Cell]
model.py: SparseCausalSelfAttention.forward


In [29]:
# [Code Cell]
append_module('model.py', r'''    def forward(self, hidden: Tensor, key_valid: Tensor | None) -> Tensor:
        positions = torch.arange(hidden.shape[1], device=hidden.device)
        query, key, value = self.project(hidden, positions)
        return self.attend(query, (key, value), positions, key_valid)


''')


[Markdown Cell]
model.py: SwiGLU


In [30]:
# [Code Cell]
append_module('model.py', r'''class SwiGLU(nn.Module):
    """hidden[B,T,D] -> Wdown(SiLU(Wgate h) * Wup h)[B,T,D]."""

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, hidden: Tensor) -> Tensor:
        return self.down(F.silu(self.gate(hidden)) * self.up(hidden))


''')


[Markdown Cell]
model.py: DecoderBlock


In [31]:
# [Code Cell]
append_module('model.py', r'''class DecoderBlock(nn.Module):
    """RMSNorm -> sparse causal attention -> residual -> RMSNorm -> SwiGLU -> residual."""

    def __init__(self, cfg: TrainConfig, layer_index: int) -> None:
        super().__init__()
        self.attention_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.attention = SparseCausalSelfAttention(cfg, layer_index)
        self.ffn = SwiGLU(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, hidden: Tensor, key_valid: Tensor | None) -> Tensor:
        hidden = hidden + self.dropout(self.attention(self.attention_norm(hidden), key_valid))
        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden)))

    def cached(self, hidden: Tensor, positions: Tensor, previous: KeyValue | None) -> tuple[Tensor, KeyValue]:
        """Append rotated K/V, route using absolute positions, and return the updated cache."""
        query, key, value = self.attention.project(self.attention_norm(hidden), positions)
        kv = (key, value) if previous is None else (
            torch.cat((previous[0], key), dim=2), torch.cat((previous[1], value), dim=2))
        hidden = hidden + self.dropout(self.attention.attend(query, kv, positions))
        return hidden + self.dropout(self.ffn(self.ffn_norm(hidden))), kv


''')


[Markdown Cell]
model.py: DecoderCache


In [32]:
# [Code Cell]
append_module('model.py', r'''@dataclass
class DecoderCache:
    """Inference-only per-layer rotary K/V; position counts consumed input tokens.

    Retain every past K/V up to max_seq_tokens: a key outside today's window can
    still be selected by a later dilation or global landmark route.
    """
    key_values: list[KeyValue]
    position: int


''')


[Markdown Cell]
model.py: LegalSparseDecoder


In [33]:
# [Code Cell]
append_module('model.py', r'''class LegalSparseDecoder(nn.Module):
    """From-scratch local/dilated/global-landmark legal causal language model.

    forward(input_ids[B,T], attention_mask[B,T] | None, labels[B,T] | None)
      -> logits[B,T,V], or summed FP32 NLL for already-shifted labels (-100 ignored).
    Sparse routing has no learned parameters; tied embedding/head is counted once.
    """

    def __init__(self, cfg: TrainConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList(DecoderBlock(cfg, index) for index in range(cfg.n_layers))
        self.final_norm = RMSNorm(cfg.d_model, cfg.rms_epsilon)
        self.apply(self.initialize_weights)
        for layer in self.layers:
            nn.init.normal_(layer.attention.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(layer.ffn.down.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        self.parameter_count = assert_parameter_boundary(self)

    @staticmethod
    def initialize_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def hidden_states(self, input_ids: Tensor, attention_mask: Tensor | None = None) -> Tensor:
        assert input_ids.ndim == 2 and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens
        if attention_mask is not None:
            assert attention_mask.shape == input_ids.shape and attention_mask.dtype == torch.bool
            assert bool(attention_mask[:, 0].all()), "Right padding requires a valid first token"
            assert not bool((attention_mask[:, 1:] & ~attention_mask[:, :-1]).any()), "Use right padding"
        hidden = self.token_embedding(input_ids)
        for layer in self.layers:
            if self.training and self.cfg.activation_checkpointing:
                hidden = checkpoint(layer, hidden, attention_mask, use_reentrant=False)
            else:
                hidden = layer(hidden, attention_mask)
        return self.final_norm(hidden)

    def chunk_loss(self, hidden: Tensor, labels: Tensor) -> Tensor:
        logits = F.linear(hidden, self.token_embedding.weight)
        return F.cross_entropy(logits.float(), labels, ignore_index=-100, reduction="sum")

    def forward(self, input_ids: Tensor, attention_mask: Tensor | None = None,
                labels: Tensor | None = None, label_indices: Tensor | None = None) -> Tensor:
        """IDs[B,T], right-padding mask, labels -> logits or exact summed target NLL."""
        hidden = self.hidden_states(input_ids, attention_mask)
        if labels is None:
            return F.linear(hidden, self.token_embedding.weight)
        assert labels.shape == input_ids.shape
        return exact_vocabulary_loss(hidden, labels, self.token_embedding.weight,
                                     self.cfg.loss_chunk_tokens, self.training, label_indices)


    @torch.inference_mode()
    def prefill(self, input_ids: Tensor) -> tuple[Tensor, DecoderCache]:
        """Unpadded prompt[B,T] -> next-token logits[B,V] plus T-token rotary K/V."""
        assert not self.training and input_ids.ndim == 2 and 0 < input_ids.shape[1] <= self.cfg.max_seq_tokens
        hidden = self.token_embedding(input_ids)
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        key_values = []
        for layer in self.layers:
            hidden, kv = layer.cached(hidden, positions, None)
            key_values.append(kv)
        return F.linear(self.final_norm(hidden[:, -1]), self.token_embedding.weight), DecoderCache(
            key_values, input_ids.shape[1])


''')


[Markdown Cell]
model.py: decode_one_token


In [34]:
# [Code Cell]
append_module('model.py', r'''@torch.inference_mode()
def decode_one_token(model: LegalSparseDecoder, token_ids: Tensor, cache: DecoderCache) -> Tensor:
    """token_ids[B,1] at cache.position -> next-token logits[B,V]; update sparse cache."""
    assert not model.training and token_ids.ndim == 2 and token_ids.shape[1] == 1
    assert 0 < cache.position < model.cfg.max_seq_tokens
    assert len(cache.key_values) == len(model.layers)
    hidden = model.token_embedding(token_ids)
    positions = torch.tensor([cache.position], device=token_ids.device)
    for index, layer in enumerate(model.layers):
        hidden, cache.key_values[index] = layer.cached(hidden, positions, cache.key_values[index])
    cache.position += 1
    return F.linear(model.final_norm(hidden[:, 0]), model.token_embedding.weight)

# CELL: Exact supervised-token vocabulary projection in bounded chunks
''')


[Markdown Cell]
model.py: exact_vocabulary_loss


In [35]:
# [Code Cell]
append_module('model.py', r'''def exact_vocabulary_loss(hidden: Tensor, labels: Tensor, weight: Tensor,
                          chunk_tokens: int, training: bool,
                          label_indices: Tensor | None = None) -> Tensor:
    """[B,T,D], shifted labels[B,T] -> summed FP32 NLL with the same objective.

    Collation computes supervised flat indices on CPU, avoiding CUDA nonzero
    synchronization. Only PAD/prompt positions with label=-100 are omitted.
    Every real target remains supervised. Recompute bounded logits in backward.
    """
    hidden, targets = hidden.flatten(0, 1), labels.flatten()
    if label_indices is not None:
        hidden = hidden.index_select(0, label_indices)
        targets = targets.index_select(0, label_indices)
    if targets.numel() == 0:
        # A final distributed sentinel contributes zero but visits every layer.
        return hidden.sum() * 0.0

    def project_loss(values: Tensor, target: Tensor, embedding: Tensor) -> Tensor:
        logits = F.linear(values, embedding)
        return F.cross_entropy(logits.float(), target, ignore_index=-100, reduction="sum")

    losses = []
    for start in range(0, targets.numel(), chunk_tokens):
        values, target = hidden[start:start + chunk_tokens], targets[start:start + chunk_tokens]
        if training and torch.is_grad_enabled():
            losses.append(checkpoint(project_loss, values, target, weight, use_reentrant=False))
        else:
            losses.append(project_loss(values, target, weight))
    return torch.stack(losses).sum()
''')


[Markdown Cell]
## 4. Exact causal sparse connectivity
Let $t$ be a query's absolute position within the current document window,
$W=128$, $J=16$, $G=256$, and $\delta_l=64\,2^{l\bmod4}$ in layer $l$.
Define the allowed key **set**, clipping to $0\le s\le t$:
\[
\mathcal K_l(t)=\{t-i:0\le i<W\}
\cup\{t-W-j\delta_l:0\le j<J\}
\cup\{mG:m\ge0,\ mG\le t\}.
\]
The local window includes the current key. Dilated links begin immediately
outside it; deeper layers alternate wider distances. Periodic landmark keys
provide shared distant context, including position 0. Their own hidden states
are computed causally too. These are ordinary sequence tokens, with no inserted
global tokens or future-reading global queries. This is a custom causal design
inspired by [Longformer's local/dilated/global patterns](https://arxiv.org/abs/2004.05150).

Sort selected indices and mask duplicates: an overlapping local/dilated/landmark
key must appear only once in the softmax. Invalid early-position candidates get
safe gather indices and remain masked. This helper allocates $Q_c\times K$ indices,
never $T\times T$. Padding is applied to the selected keys afterwards.


[Markdown Cell]
## 5. Sparse computation and memory bounds
\[
a_{t,s}=\frac{\exp(R_tq_t\cdot R_sk_s/\sqrt{d_h})}
{\sum_{u\in\mathcal K_l(t)}\exp(R_tq_t\cdot R_uk_u/\sqrt{d_h})},\quad
o_t=\sum_{s\in\mathcal K_l(t)}a_{t,s}v_s.
\]
Gather selected K/V for at most $Q_c=32$ queries, then reshape each query as
an independent SDPA item. Boolean True means a legal key; `is_causal=False`
is correct because the index helper has already enforced absolute causality.
This works for rectangular cached decoding too. PyTorch selects a supported
kernel; the implementation does not require FlashAttention or a custom extension.

Candidate count is bounded by $K=W+J+\lceil T/G\rceil$: at most **152** at
$T=2048$, compared with 2048 dense keys per query. Gathered K/V peak memory is
$O(BHQ_cKd_h)$; attention arithmetic is $O(BHTKd_h)$, plus index sorting.
Fixed $G$ leaves an $O(T^2/G)$ landmark term as context grows: this is **not an
unbounded linear-complexity claim**. The configured context is capped at 2048.
Small query chunks and non-reentrant attention checkpointing discard gathered
activations and recompute them in backward. Indexing and recomputation add work;
measure actual T4 throughput instead of assuming a speedup over fused dense SDPA.
[SDPA API](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention),
[checkpoint API](https://docs.pytorch.org/docs/stable/checkpoint).


[Markdown Cell]
## 6. Sparse decoder residual block
\[
h'=h+\operatorname{Dropout}(\operatorname{SparseAttn}(\operatorname{RMSNorm}(h))),
\quad \hat h=\operatorname{RMSNorm}(h'),
\quad h''=h'+\operatorname{Dropout}(W_d(\operatorname{SiLU}(W_g\hat h)\odot W_u\hat h)).
\]
All projections are bias-free; normalization vectors are
learned. Each block receives a layer index to choose its dilation. Cached keys
already contain their RoPE rotation. Preserve all past K/V up to the context
limit, because an old key can enter a later dilated route. K/V storage remains
$O(BLTD)$ even though attention computation is sparse.


[Markdown Cell]
## 7. Complete model and exact vocabulary loss
Inputs and already-shifted labels are $[B,T]$. Inference returns $[B,T,V]$ logits;
training returns summed FP32 cross-entropy, ignoring labels equal to -100.
Layer and attention-chunk checkpointing reduce activation retention. Project the
tied vocabulary in bounded chunks of up to 512 supervised positions, checkpointing each chunk too.
This preserves the exact objective without retaining a complete $BTV$ vocabulary
tensor for backward. QA compares dense/chunk losses and parameter gradients.
No pretrained weights or pretrained tokenizer are loaded.


[Markdown Cell]
## 8. Cache semantics and train/inference agreement
Prefill the unpadded prompt using the same sparse set $\mathcal K_l(t)$ as training.
A new query uses its true absolute position, with all past K/V retained.
Neither the local window nor dilation is re-centered at zero during decoding;
landmark positions remain absolute. Clear caches for each new prompt. The context
cap is enforced before appending a token. Verify cached/full logits across both
the 128-token window and the landmark at position 256.


[Markdown Cell]
## 9. Verify the preprocessing export
Check schema, uint16 byte order, 32,768 vocabulary, six special IDs, SHA-256 receipts, token ranges, the 35/35/15/15 training quotas, and equal validation cache quotas. Both binaries are scanned in bounded memmap slices. The tokenizer is reused exactly; no HF access is required.


[Markdown Cell]
data.py: imports and module constants


In [36]:
# [Code Cell]
append_module('data.py', r'''from functools import partial
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SPECIAL_TOKENS, SOURCES


''')


[Markdown Cell]
data.py: sha256_file


In [37]:
# [Code Cell]
append_module('data.py', r'''def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 << 20):
            digest.update(block)
    return digest.hexdigest()


''')


[Markdown Cell]
data.py: strict_object


In [38]:
# [Code Cell]
append_module('data.py', r'''def strict_object(pairs: list[tuple]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result


''')


[Markdown Cell]
data.py: read_json


In [39]:
# [Code Cell]
append_module('data.py', r'''def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"), object_pairs_hook=strict_object,
                      parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))


''')


[Markdown Cell]
data.py: verify_data_contract


In [40]:
# [Code Cell]
append_module('data.py', r'''def verify_data_contract(cfg: TrainConfig) -> dict:
    """Verify bounded-memory binaries, tokenizer identity, and published receipts."""
    root = Path(cfg.data_dir)
    metadata = read_json(root / "metadata.json")
    assert metadata["status"] == "verified" and metadata["schema_version"] == 1
    assert metadata["dtype"] == "<u2" and metadata["vocab_size"] == cfg.vocab_size
    assert metadata["train_tokens"] % 100 == 0 and metadata["val_tokens"] % 4 == 0
    assert metadata["training"]["source_tokens"] == dict(zip(
        SOURCES, [metadata["train_tokens"] * percent // 100 for percent in (35, 35, 15, 15)]))
    assert all(metadata["validation"][source]["cached_tokens"] == metadata["val_tokens"] // 4
               for source in SOURCES)
    required = ("train.bin", "val.bin", "tokenizer.json", "train_documents.jsonl",
                "validation_documents.jsonl", "validation_pairs.jsonl")
    for name in tqdm(required, desc="Verify preprocessing receipts"):
        path, receipt = root / name, metadata["artifacts"][name]
        assert path.stat().st_size == receipt["bytes"], f"Size mismatch: {name}"
        assert sha256_file(path) == receipt["sha256"], f"Checksum mismatch: {name}"
    tokenizer = Tokenizer.from_file(str(root / "tokenizer.json"))
    ids = {token: tokenizer.token_to_id(token) for token in SPECIAL_TOKENS}
    assert tokenizer.get_vocab_size() == 32_768
    assert ids == metadata["special_token_ids"] == dict(zip(SPECIAL_TOKENS, range(6)))
    for name, count in (("train.bin", metadata["train_tokens"]), ("val.bin", metadata["val_tokens"])):
        assert count > 0 and (root / name).stat().st_size == count * 2
        cache = np.memmap(root / name, dtype="<u2", mode="r", shape=(count,))
        for start in range(0, count, 1_000_000):
            assert int(cache[start:start + 1_000_000].max()) < cfg.vocab_size
        del cache
    identity = {name: metadata["artifacts"][name]["sha256"] for name in required}
    identity["metadata.json"] = sha256_file(root / "metadata.json")
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    return {"fingerprint": fingerprint, "metadata": metadata, "special_ids": ids}


''')


[Markdown Cell]
data.py: prepare_indices


In [41]:
# [Code Cell]
append_module('data.py', r'''def prepare_indices(cfg: TrainConfig, contract: dict) -> dict:
    """Index document-local next-token windows; retain every within-document transition.

    Adjacent windows share one input context token. Each target is supervised once
    per document pass, without predicting across independent document boundaries.
    """
    root, run = Path(cfg.data_dir), Path(cfg.run_dir)
    index_dir = run / "indices"
    index_dir.mkdir(parents=True, exist_ok=True)
    offsets, lengths, counts, source_indices = [], [], [], []
    train_hashes, validation_hashes = set(), set()
    source_tokens, target_tokens, dropped_tokens = Counter(), Counter(), Counter()
    cursor = 0
    with (root / "train_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            row = json.loads(line, object_pairs_hook=strict_object)
            offset, length, source = row["cache_offset"], row["cache_length"], row["source"]
            assert source in SOURCES and offset == cursor and length > 0
            assert row["sha256"] not in train_hashes
            train_hashes.add(row["sha256"])
            predictions = length - 1
            count = (predictions + cfg.train_seq_tokens - 1) // cfg.train_seq_tokens
            if count:
                offsets.append(offset); lengths.append(length); counts.append(count)
                source_indices.append(SOURCES.index(source))
            source_tokens[source] += length
            target_tokens[source] += predictions
            dropped_tokens[source] += int(length == 1)
            cursor += length
    assert cursor == contract["metadata"]["train_tokens"]
    assert dict(source_tokens) == contract["metadata"]["training"]["source_tokens"]
    with (root / "validation_documents.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            digest = json.loads(line)["sha256"]
            assert digest not in validation_hashes
            validation_hashes.add(digest)
    assert train_hashes.isdisjoint(validation_hashes)
    cumulative = np.cumsum(np.asarray(counts, dtype=np.int64))
    assert len(cumulative) and int(cumulative[-1]) >= cfg.world_size * cfg.micro_batch_size
    train_path = index_dir / "train_index.npz"
    temporary = train_path.with_suffix(".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, offsets=np.array(offsets, dtype=np.int64),
                            lengths=np.array(lengths, dtype=np.int64), cumulative=cumulative,
                            sources=np.array(source_indices, dtype=np.uint8))
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, train_path)
    validation = prepare_validation_index(cfg, validation_hashes, index_dir)
    report = {"train_examples": int(cumulative[-1]), "cached_source_tokens": dict(source_tokens),
              "eligible_target_tokens": dict(target_tokens), "one_token_documents_dropped": dict(dropped_tokens),
              "global_batch_tail_max_examples": 0, "full_window_coverage": True,
              "validation": validation, "data_fingerprint": contract["fingerprint"]}
    (index_dir / "report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report


''')


[Markdown Cell]
data.py: tokenize_pair


In [42]:
# [Code Cell]
append_module('data.py', r'''def tokenize_pair(tokenizer: Tokenizer, prompt: str, reference: str,
                  cfg: TrainConfig) -> tuple[list[int], list[int], bool, bool]:
    """Encode prompt and target separately; report BPE crops, preserve the prompt tail."""
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    target_ids = tokenizer.encode(" " + reference, add_special_tokens=False).ids
    prompt_cropped = len(prompt_ids) > cfg.max_prompt_tokens - 1
    target_cropped = len(target_ids) + 1 > cfg.max_target_tokens
    prefix = [1] + prompt_ids[-(cfg.max_prompt_tokens - 1):]
    target = target_ids[:cfg.max_target_tokens] if target_cropped else target_ids + [2]
    assert prefix and target and len(prefix) + len(target) - 1 <= cfg.max_seq_tokens
    return prefix, target, prompt_cropped, target_cropped


''')


[Markdown Cell]
data.py: prepare_validation_index


In [43]:
# [Code Cell]
append_module('data.py', r'''def prepare_validation_index(cfg: TrainConfig, allowed_hashes: set[str], index_dir: Path) -> dict:
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    offsets, identifiers, statistics = [], set(), Counter()
    with (Path(cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
        while True:
            offset = stream.tell()
            line = stream.readline()
            if not line:
                break
            row = json.loads(line, object_pairs_hook=strict_object)
            assert row["id"] not in identifiers and row["document_sha256"] in allowed_hashes
            assert len(row["prompt"].split()) == 512 and len(row["reference"].split()) == 128
            assert row["source"] in SOURCES
            identifiers.add(row["id"]); offsets.append(offset)
            _, _, prompt_crop, target_crop = tokenize_pair(tokenizer, row["prompt"], row["reference"], cfg)
            statistics[row["source"]] += 1
            statistics["prompt_truncations"] += int(prompt_crop)
            statistics["target_truncations"] += int(target_crop)
    assert offsets and all(statistics[source] for source in SOURCES)
    path = index_dir / "validation_offsets.npy"
    with path.with_suffix(".partial").open("wb") as stream:
        np.save(stream, np.asarray(offsets, dtype=np.int64), allow_pickle=False)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(path.with_suffix(".partial"), path)
    return dict(statistics, examples=len(offsets))


''')


[Markdown Cell]
data.py: ContinuationDataset


In [44]:
# [Code Cell]
append_module('data.py', r'''class ContinuationDataset(Dataset):
    """One document window -> input_ids[T], shifted labels[T]; all transitions train."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg, self.cache = cfg, None
        with np.load(Path(cfg.run_dir) / "indices/train_index.npz", allow_pickle=False) as index:
            self.offsets, self.lengths = index["offsets"].copy(), index["lengths"].copy()
            self.cumulative = index["cumulative"].copy()

    def __len__(self) -> int:
        return int(self.cumulative[-1])

    def __getstate__(self) -> dict:
        return dict(self.__dict__, cache=None)

    def __getitem__(self, index: int) -> dict:
        if index == -1:
            return {"input_ids": [1], "labels": [-100]}
        assert 0 <= index < len(self)
        if self.cache is None:
            self.cache = np.memmap(Path(self.cfg.data_dir) / "train.bin", dtype="<u2", mode="r")
        document = int(np.searchsorted(self.cumulative, index, side="right"))
        previous_count = int(self.cumulative[document - 1]) if document else 0
        within = (index - previous_count) * self.cfg.train_seq_tokens
        length = min(self.cfg.train_seq_tokens + 1, int(self.lengths[document]) - within)
        start = int(self.offsets[document]) + within
        tokens = np.asarray(self.cache[start:start + length], dtype=np.int64)
        assert 2 <= len(tokens) <= self.cfg.train_seq_tokens + 1
        return {"input_ids": tokens[:-1], "labels": tokens[1:]}


''')


[Markdown Cell]
data.py: ValidationDataset


In [45]:
# [Code Cell]
append_module('data.py', r'''class ValidationDataset(Dataset):
    """JSONL prompt/continuation teacher forcing; only reference tokens affect NLL."""

    def __init__(self, cfg: TrainConfig) -> None:
        self.cfg, self.tokenizer = cfg, None
        self.offsets = np.load(Path(cfg.run_dir) / "indices/validation_offsets.npy", allow_pickle=False)

    def __len__(self) -> int:
        return len(self.offsets)

    def __getstate__(self) -> dict:
        return dict(self.__dict__, tokenizer=None)

    def __getitem__(self, index: int) -> dict:
        if self.tokenizer is None:
            self.tokenizer = Tokenizer.from_file(str(Path(self.cfg.data_dir) / "tokenizer.json"))
        with (Path(self.cfg.data_dir) / "validation_pairs.jsonl").open("rb") as stream:
            stream.seek(int(self.offsets[index]))
            row = json.loads(stream.readline())
        prefix, target, _, _ = tokenize_pair(self.tokenizer, row["prompt"], row["reference"], self.cfg)
        return {"input_ids": (prefix + target)[:-1],
                "labels": [-100] * (len(prefix) - 1) + target}


''')


[Markdown Cell]
data.py: collate_continuations


In [46]:
# [Code Cell]
append_module('data.py', r'''def collate_continuations(rows: list[dict]) -> dict[str, torch.Tensor]:
    """Pad input IDs with PAD=0; ignored labels=-100; valid keys=True."""
    length = max(len(row["input_ids"]) for row in rows)
    input_ids = torch.zeros((len(rows), length), dtype=torch.long)
    labels = torch.full_like(input_ids, -100)
    attention_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for index, row in enumerate(rows):
        count = len(row["input_ids"])
        assert count == len(row["labels"]) and count > 0
        input_ids[index, :count] = torch.as_tensor(row["input_ids"])
        labels[index, :count] = torch.as_tensor(row["labels"])
        attention_mask[index, :count] = True
    label_indices = torch.nonzero(labels.flatten() != -100, as_tuple=False).flatten()
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "label_indices": label_indices}



''')


[Markdown Cell]
data.py: EpochBatchSampler


In [47]:
# [Code Cell]
append_module('data.py', r'''class EpochBatchSampler(Sampler):
    """Seeded global permutation; every real window appears once on exactly one rank.

    Pad only the final global batch with -1 sentinels, never repeated documents.
    Sentinel labels are all ignored, so global token weighting stays exact.
    Resume jumps directly to the saved batch without consuming earlier data.
    """
    def __init__(self, size: int, batch_size: int, rank: int, world_size: int,
                 seed: int, epoch: int, start_batch: int = 0) -> None:
        self.batch_size, self.rank, self.world_size = batch_size, rank, world_size
        global_batch = world_size * batch_size
        self.num_batches = (size + global_batch - 1) // global_batch
        assert size > 0 and 0 <= start_batch <= self.num_batches and 0 <= rank < world_size
        self.start_batch = start_batch
        generator = torch.Generator().manual_seed(seed + epoch)
        order = torch.randperm(size, generator=generator)
        padding = self.num_batches * global_batch - size
        if padding:
            order = torch.cat((order, torch.full((padding,), -1, dtype=torch.long)))
        self.batches = order.reshape(self.num_batches, world_size, batch_size)[:, rank]

    def __iter__(self):
        for index in range(self.start_batch, self.num_batches):
            yield self.batches[index].tolist()

    def __len__(self) -> int:
        return self.num_batches - self.start_batch
''')


[Markdown Cell]
Index document offsets and window counts. Consecutive windows share one input token so every within-document transition is supervised once. Never predict across documents. One-token documents have no target and are reported. Training and validation document hashes must be disjoint. Only compact indices are written to working storage.


[Markdown Cell]
Read all complete 512-word/128-word JSONL pairs, even if the last val.bin slice was cropped for its exact quota. Encode prompt and reference separately; labels ignore the prompt. Long prompts keep their recent tail; targets receive EOS only when the full reference fits. Report prompt and target truncations explicitly.


[Markdown Cell]
Training uses $x=[t_0,\ldots,t_{T-1}]$ and $y=[t_1,\ldots,t_T]$.
Only genuine cached BOS/EOS tokens are used; arbitrary window edges get no
invented EOS. Validation places reference tokens after the prompt, masks prompt
labels, and uses causality to prevent future-reference leakage. Each process
opens its read-only memmap lazily; worker serialization drops live handles.


[Markdown Cell]
Right-pad input IDs with PAD=0, labels with -100, and mark valid attention keys. A seeded global permutation supplies disjoint rank batches without duplicate padding. The sampler resumes directly at `next_batch`; every real window is included once. Ignored sentinels fill the final distributed batch without duplicating real examples.


[Markdown Cell]
## 10. Checkpoint integrity and reproducibility
Fingerprint the data, learning settings, runtime source, and PyTorch version. Incompatible state fails explicitly. Output paths and time/storage limits may change for an attached saved run. A Linux advisory lock prevents concurrent trainers from overwriting one run.


[Markdown Cell]
state.py: imports and module constants


In [48]:
# [Code Cell]
append_module('state.py', r'''from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import hashlib
import json
import math
import os
import random
import shutil
import numpy as np
import torch
import torch.distributed as dist
from .config import TrainConfig, RUNTIME_VERSION
from .data import sha256_file, read_json


''')


[Markdown Cell]
state.py: distributed


In [49]:
# [Code Cell]
append_module('state.py', r'''def distributed() -> bool:
    return dist.is_available() and dist.is_initialized()


''')


[Markdown Cell]
state.py: atomic_json


In [50]:
# [Code Cell]
append_module('state.py', r'''def atomic_json(path: Path, payload: dict) -> None:
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n"); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)


''')


[Markdown Cell]
state.py: runtime_fingerprint


In [51]:
# [Code Cell]
append_module('state.py', r'''def runtime_fingerprint(cfg: TrainConfig, contract: dict) -> str:
    """Paths and time/space limits may change; learning and data choices may not."""
    choices = asdict(cfg)
    for key in ("data_dir", "run_dir", "resume_dir", "initial_checkpoint", "max_run_seconds", "min_free_bytes",
                "max_checkpoint_bytes", "generation_examples_per_source", "generation_token_limit"):
        choices.pop(key)
    sources = {path.name: sha256_file(path) for path in sorted(Path(__file__).parent.glob("*.py"))}
    payload = {"version": RUNTIME_VERSION, "config": choices,
               "data": contract["fingerprint"], "sources": sources,
               "torch": str(torch.__version__)}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


''')


[Markdown Cell]
state.py: exclusive_run_lock


In [52]:
# [Code Cell]
append_module('state.py', r'''@contextmanager
def exclusive_run_lock(run_dir: Path):
    """Linux advisory lock is released by the OS even after a killed kernel."""
    import fcntl  # Kaggle/Linux only; production does not silently fall back.
    run_dir.mkdir(parents=True, exist_ok=True)
    with (run_dir / ".training.lock").open("a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Another Model D training process owns this run directory.") from exc
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


''')


[Markdown Cell]
state.py: seed_everything


In [53]:
# [Code Cell]
append_module('state.py', r'''def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


''')


[Markdown Cell]
state.py: capture_rng


In [54]:
# [Code Cell]
append_module('state.py', r'''def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {"python": random.getstate(), "torch_cpu": torch.get_rng_state(),
            "numpy": {"kind": numpy_state[0], "keys": torch.tensor(numpy_state[1].astype(np.int64)),
                      "position": numpy_state[2], "has_gauss": numpy_state[3],
                      "cached_gaussian": numpy_state[4]},
            "torch_cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


''')


[Markdown Cell]
state.py: restore_rng


In [55]:
# [Code Cell]
append_module('state.py', r'''def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state["python"]); torch.set_rng_state(state["torch_cpu"])
    ns = state["numpy"]
    np.random.set_state((ns["kind"], ns["keys"].numpy().astype(np.uint32), ns["position"],
                         ns["has_gauss"], ns["cached_gaussian"]))
    if device.type == "cuda":
        torch.cuda.set_rng_state(state["torch_cuda"], device)


''')


[Markdown Cell]
state.py: cpu_tree


In [56]:
# [Code Cell]
append_module('state.py', r'''def cpu_tree(value):
    """Own a CPU snapshot, retaining only tensors and weights-only-safe primitives."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value


''')


[Markdown Cell]
state.py: CheckpointManager


In [57]:
# [Code Cell]
append_module('state.py', r'''class CheckpointManager:
    """Two full recovery slots plus one full best checkpoint, each with SHA-256.

    The inactive slot is replaced first; its receipt commits last. A crash
    between those writes leaves the previously committed slot recoverable.
    Every full checkpoint includes optimizer, scaler, scheduler, and cursor.
    """

''')


[Markdown Cell]
state.py: CheckpointManager.__init__


In [58]:
# [Code Cell]
append_module('state.py', r'''    def __init__(self, cfg: TrainConfig, fingerprint: str) -> None:
        self.cfg, self.fingerprint, self.generation = cfg, fingerprint, 0
        self.directory = Path(cfg.run_dir) / "checkpoints"
        self.directory.mkdir(parents=True, exist_ok=True)

''')


[Markdown Cell]
state.py: CheckpointManager.latest


In [59]:
# [Code Cell]
append_module('state.py', r'''    def latest(self) -> tuple[Path, dict] | None:
        roots = [self.directory]
        if self.cfg.resume_dir:
            roots.append(Path(self.cfg.resume_dir))
        candidates, seen, problems = [], False, []
        for root in roots:
            for name in ("slot_0.pt", "slot_1.pt", "best.pt"):
                path, receipt_path = root / name, root / (name + ".json")
                if not path.exists() and not receipt_path.exists():
                    continue
                seen = True
                try:
                    receipt = read_json(receipt_path)
                    assert isinstance(receipt, dict)
                    assert all(type(receipt[key]) is int for key in ("bytes", "step", "generation", "epoch"))
                    assert receipt["bytes"] > 0 and receipt["step"] >= 0
                    assert receipt["generation"] > 0 and receipt["epoch"] >= 0
                    assert path.stat().st_size == receipt["bytes"]
                    assert sha256_file(path) == receipt["sha256"]
                    candidates.append((path, receipt))
                except (OSError, ValueError, TypeError, KeyError, AssertionError) as exc:
                    problems.append(f"{name}: {type(exc).__name__}")
        if seen and not candidates:
            raise RuntimeError(f"No intact checkpoint; preserve files and inspect receipts. {problems}")
        if not candidates:
            if self.cfg.resume_dir:
                raise FileNotFoundError("resume_dir contains no committed checkpoints.")
            return None
        chosen = max(candidates, key=lambda item: (item[1]["step"], item[1]["generation"]))
        if chosen[1]["fingerprint"] != self.fingerprint:
            raise RuntimeError("Checkpoint data/configuration/runtime differs. Use a fresh run directory.")
        self.generation = chosen[1]["generation"]
        if problems and (not distributed() or dist.get_rank() == 0):
            print(f"Ignoring incomplete checkpoints: {problems}; recovered step {chosen[1]['step']}", flush=True)
        return chosen

''')


[Markdown Cell]
state.py: CheckpointManager.ensure_space


In [60]:
# [Code Cell]
append_module('state.py', r'''    def ensure_space(self, estimated_bytes: int) -> None:
        owned = sum(path.stat().st_size for path in self.directory.iterdir() if path.is_file())
        free = shutil.disk_usage(self.directory).free
        if free < estimated_bytes + self.cfg.min_free_bytes:
            raise OSError("Insufficient disk space for an atomic checkpoint and free-space reserve.")
        if owned + estimated_bytes > self.cfg.max_checkpoint_bytes:
            raise OSError("Checkpoint storage cap reached. Inspect this run's checkpoint files.")

''')


[Markdown Cell]
state.py: CheckpointManager.restore


In [61]:
# [Code Cell]
append_module('state.py', r'''    def restore(self, model, optimizer, scaler, scheduler, device: torch.device) -> dict | None:
        chosen = self.latest()
        if chosen is None:
            return None
        path, receipt = chosen
        payload = torch.load(path, map_location="cpu", weights_only=True)
        assert payload["fingerprint"] == self.fingerprint
        assert payload["step"] == receipt["step"] and payload["epoch"] == receipt["epoch"]
        assert payload["world_size"] == self.cfg.world_size
        model.load_state_dict(payload["model"], strict=True)
        optimizer.load_state_dict(payload["optimizer"])
        scaler.load_state_dict(payload["scaler"])
        scheduler.load_state_dict(payload["scheduler"])
        assert scheduler.steps == payload["cursor"]["optimizer_steps"]
        return payload


''')


[Markdown Cell]
state.py: CheckpointManager.save


In [62]:
# [Code Cell]
append_module('state.py', r'''    def save(self, model, optimizer, scaler, scheduler, cursor: dict,
             device: torch.device, rank: int, best: bool = False) -> None:
        local_rng = capture_rng(device)
        rng_states = [None] * dist.get_world_size() if distributed() else [local_rng]
        if distributed():
            dist.all_gather_object(rng_states, local_rng)
        outcome = [None]
        if rank == 0:
            try:
                self.generation += 1
                path = self.directory / f"slot_{self.generation % 2}.pt"
                temporary = path.with_name(path.name + ".partial")
                # FP32 weights + Adam's two FP32 moments + serialization headroom.
                estimated = sum(p.numel() for p in model.parameters()) * 12 + (32 << 20)
                self.ensure_space(estimated)
                payload = {"version": RUNTIME_VERSION, "fingerprint": self.fingerprint,
                           "world_size": self.cfg.world_size, "config": asdict(self.cfg),
                           "epoch": int(cursor["epoch"]), "step": int(cursor["global_step"]),
                           "model": cpu_tree(model.state_dict()),
                           "optimizer": cpu_tree(optimizer.state_dict()), "scaler": scaler.state_dict(),
                           "scheduler": scheduler.state_dict(), "cursor": dict(cursor),
                           "rng_by_rank": rng_states}
                with temporary.open("wb") as stream:
                    torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
                receipt = {"bytes": temporary.stat().st_size, "sha256": sha256_file(temporary),
                           "fingerprint": self.fingerprint, "generation": self.generation,
                           "epoch": payload["epoch"], "step": payload["step"],
                           "validation_nll": (cursor["best_validation_nll"]
                                              if math.isfinite(cursor["best_validation_nll"]) else None)}
                os.replace(temporary, path)
                atomic_json(path.with_name(path.name + ".json"), receipt)
                if best:
                    self.publish_best(path, receipt)
                outcome[0] = {"ok": True, "generation": self.generation}
            except Exception as exc:
                outcome[0] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        if distributed():
            dist.broadcast_object_list(outcome, src=0)
        if not outcome[0]["ok"]:
            raise RuntimeError(f"Checkpoint failed: {outcome[0]['error']}")
        self.generation = outcome[0]["generation"]

''')


[Markdown Cell]
state.py: CheckpointManager.publish_best


In [63]:
# [Code Cell]
append_module('state.py', r'''    def publish_best(self, source: Path, receipt: dict) -> None:
        """Hard-link the complete immutable snapshot; copy if links are unavailable."""
        path = self.directory / "best.pt"
        temporary = self.directory / "best.pt.partial"
        if temporary.exists():
            temporary.unlink()
        try:
            os.link(source, temporary)
        except OSError:
            self.ensure_space(source.stat().st_size)
            shutil.copyfile(source, temporary)
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
        os.replace(temporary, path)
        atomic_json(path.with_name(path.name + ".json"), receipt)


''')


[Markdown Cell]
state.py: WarmupCosineScheduler


In [64]:
# [Code Cell]
append_module('state.py', r'''class WarmupCosineScheduler:
    """LR advances only after a successful optimizer update, including resume."""

    def __init__(self, optimizer, total_steps: int, warmup_steps: int,
                 peak_lr: float, minimum_ratio: float) -> None:
        self.optimizer, self.total_steps = optimizer, total_steps
        self.warmup_steps = min(warmup_steps, max(0, total_steps - 1))
        self.peak_lr, self.minimum_ratio, self.steps = peak_lr, minimum_ratio, 0
        self.apply_lr()

    def apply_lr(self) -> None:
        if self.warmup_steps and self.steps < self.warmup_steps:
            ratio = (self.steps + 1) / self.warmup_steps
        else:
            denominator = max(1, self.total_steps - self.warmup_steps - 1)
            progress = min(1.0, max(0.0, (self.steps - self.warmup_steps) / denominator))
            ratio = self.minimum_ratio + (1 - self.minimum_ratio) * (1 + math.cos(math.pi * progress)) / 2
        for group in self.optimizer.param_groups:
            group["lr"] = self.peak_lr * ratio

    def step(self) -> None:
        self.steps += 1
        self.apply_lr()

    def state_dict(self) -> dict:
        return {key: getattr(self, key) for key in (
            "total_steps", "warmup_steps", "peak_lr", "minimum_ratio", "steps")}

    def load_state_dict(self, state: dict) -> None:
        for key in ("total_steps", "warmup_steps", "peak_lr", "minimum_ratio"):
            assert state[key] == getattr(self, key), f"Scheduler mismatch: {key}"
        self.steps = state["steps"]
        self.apply_lr()
''')


[Markdown Cell]
state.py: verify_initial_checkpoint


In [65]:
# [Code Cell]
append_module('state.py', r'''def verify_initial_checkpoint(cfg: TrainConfig, contract: dict) -> None:
    """Check the original data identity and selected parent SHA, without changing it."""
    if not cfg.initial_checkpoint:
        return
    if cfg.parent_data_fingerprint != contract["fingerprint"]:
        raise ValueError("Initial checkpoint's tokenizer/data fingerprint does not match this stage")
    path = Path(cfg.initial_checkpoint)
    receipt = read_json(path.with_name(path.name + ".json"))
    if receipt.get("sha256") != cfg.initial_checkpoint_sha256:
        raise ValueError("Initial checkpoint receipt differs from the recorded parent")
    if path.stat().st_size != receipt["bytes"] or sha256_file(path) != cfg.initial_checkpoint_sha256:
        raise ValueError("Initial checkpoint size/SHA verification failed")


''')


[Markdown Cell]
state.py: load_initial_weights


In [66]:
# [Code Cell]
append_module('state.py', r'''def load_initial_weights(model, cfg: TrainConfig, contract: dict) -> dict | None:
    """Strictly load only parent weights; fresh optimizer/scaler/scheduler are retained.

    Memory-map the parent full checkpoint so Adam moments never need a RAM copy.
    The actual loaded FP32 weights must be finite and have the exact architecture.
    """
    if not cfg.initial_checkpoint:
        return None
    verify_initial_checkpoint(cfg, contract)
    path = Path(cfg.initial_checkpoint)
    before = path.stat()
    receipt = read_json(path.with_name(path.name + ".json"))
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    if payload["fingerprint"] != receipt["fingerprint"] or payload["epoch"] != receipt["epoch"] or payload["step"] != receipt["step"]:
        raise ValueError("Parent payload and checkpoint receipt disagree")
    if payload["cursor"].get("optimizer_steps", 0) <= 0:
        raise ValueError("Parent checkpoint contains no successful optimizer updates")
    for name, value in payload["model"].items():
        if value.is_floating_point():
            if value.dtype != torch.float32:
                raise ValueError(f"Expected FP32 parent master weights: {name}")
            flat = value.reshape(-1)
            for start in range(0, flat.numel(), 1_000_000):
                if not bool(torch.isfinite(flat[start:start + 1_000_000]).all()):
                    raise ValueError(f"Non-finite parent weights: {name}")
    model.load_state_dict(payload["model"], strict=True)
    after = path.stat()
    if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
        raise ValueError("Parent checkpoint changed while loading")
    result = {"parent_epoch": payload["epoch"], "parent_step": payload["step"],
              "parent_sha256": cfg.initial_checkpoint_sha256, "strict_parent_weights": "passed"}
    del payload
    return result
''')


[Markdown Cell]
Save Python, NumPy, CPU Torch, and each GPU rank's CUDA RNG state. Dedicated loader and sampler generators keep prefetch and restart from perturbing dropout. Load tensors and primitives with `weights_only=True`. Saved-cursor replay is exact; CUDA bitwise identity also depends on kernels and the runtime.


[Markdown Cell]
Retain two full recovery slots and one full best checkpoint. Receipts verify SHA-256 and commit last; an incomplete latest write falls back to an intact older slot. Automatically load the newest compatible checkpoint in the working run or the explicitly attached `resume_dir`.


[Markdown Cell]
Rank 0 saves model, optimizer, scaler, scheduler, epoch/step integers, next batch, epoch statistics, and all rank RNG states. Save only after a complete accumulation group. Notify every rank of success or failure. Check disk space and the 6GiB owned-file cap before serialization. Best snapshots also contain full recovery state.


[Markdown Cell]
Warm up to the peak LR and decay toward 10% of it with cosine scheduling. Advance only after a successful optimizer update. Restore scheduler position and LR with the optimizer; an AMP overflow consumes the group without advancing either.


[Markdown Cell]
## 11. Distributed causal optimization
\[
\mathcal L_{\rm train}=-N^{-1}\sum_{d}\sum_t\log p_\theta(t_{t+1}\mid t_{\le t}).
\]
Train on all document transitions. Validation NLL measures only reference tokens
conditioned on the held-out prompt, making it comparable to Model B's continuation
metric. AdamW decays matrix weights once and excludes normalization vectors.
FP16 autocast accelerates projections; FP32 cross-entropy protects the softmax.


[Markdown Cell]
Validate parent provenance before loading existing trained weights


[Markdown Cell]
train.py: imports and module constants


In [67]:
# [Code Cell]
append_module('train.py', r'''import argparse
from contextlib import nullcontext
from datetime import timedelta
from functools import partial
from pathlib import Path
import json
import math
import os
import signal
import socket
import time
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from .config import TrainConfig, assert_parameter_boundary, resolve_training_config
from .data import (verify_data_contract, prepare_indices, ContinuationDataset,
                   ValidationDataset, EpochBatchSampler, collate_continuations)
from .model import LegalSparseDecoder
from .state import (CheckpointManager, WarmupCosineScheduler, atomic_json, distributed,
                    exclusive_run_lock, runtime_fingerprint, seed_everything, restore_rng, load_initial_weights, verify_initial_checkpoint)


''')


[Markdown Cell]
train.py: build_optimizer


In [68]:
# [Code Cell]
append_module('train.py', r'''def build_optimizer(model: nn.Module, cfg: TrainConfig):
    """Decay matrix weights once; exclude bias and normalization vectors."""
    decay, no_decay = [], []
    for parameter in model.parameters():
        if parameter.requires_grad:
            (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), eps=cfg.adam_epsilon,
        foreach=False, fused=cfg.fused_adamw and next(model.parameters()).is_cuda,
    )


''')


[Markdown Cell]
train.py: reduce_sum


In [69]:
# [Code Cell]
append_module('train.py', r'''def reduce_sum(tensor: torch.Tensor) -> torch.Tensor:
    if distributed():
        dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor


''')


[Markdown Cell]
train.py: any_rank


In [70]:
# [Code Cell]
append_module('train.py', r'''def any_rank(flag: bool, device: torch.device) -> bool:
    value = torch.tensor(int(flag), device=device, dtype=torch.int32)
    if distributed():
        dist.all_reduce(value, op=dist.ReduceOp.MAX)
    return bool(value.item())


''')


[Markdown Cell]
train.py: move_batch


In [71]:
# [Code Cell]
append_module('train.py', r'''def move_batch(batch: dict, device: torch.device) -> dict:
    """Select implicit masks on CPU and transfer pinned tensors asynchronously."""
    result = {}
    for key, value in batch.items():
        if key in {"attention_mask", "source_mask", "decoder_mask"} and bool(value.all()):
            result[key] = None
        else:
            result[key] = value.to(device, non_blocking=device.type == "cuda")
    return result


''')


[Markdown Cell]
train.py: autocast_context


In [72]:
# [Code Cell]
append_module('train.py', r'''def autocast_context(device: torch.device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()


''')


[Markdown Cell]
train.py: summed_loss


In [73]:
# [Code Cell]
append_module('train.py', r'''def summed_loss(model, batch: dict) -> torch.Tensor:
    return model(batch["input_ids"], batch["attention_mask"], labels=batch["labels"],
                 label_indices=batch.get("label_indices"))


''')


[Markdown Cell]
train.py: train_one_update


In [74]:
# [Code Cell]
append_module('train.py', r'''def train_one_update(model, optimizer, scaler, scheduler, batches: list[dict],
                     cfg: TrainConfig, device: torch.device) -> dict:
    """One token-weighted update with one distributed finite check per group.

    Accumulate device flags through all microbatches; all ranks finish the same
    DDP collective sequence before checking. No optimizer mutation occurs when
    loss is invalid; synchronized AMP overflow skips the whole update together.
    """
    world_size = dist.get_world_size() if distributed() else 1
    count = sum(int((batch["labels"] != -100).sum()) for batch in batches)
    global_tokens = int(reduce_sum(torch.tensor(count, dtype=torch.int64, device=device)).item())
    assert global_tokens > 0
    optimizer.zero_grad(set_to_none=True)
    local_loss = torch.zeros((), dtype=torch.float64, device=device)
    invalid_loss = torch.zeros((), dtype=torch.bool, device=device)
    used_lr = optimizer.param_groups[0]["lr"]
    for index, cpu_batch in enumerate(batches):
        batch = move_batch(cpu_batch, device)
        synchronize = index == len(batches) - 1
        context = model.no_sync() if isinstance(model, DDP) and not synchronize else nullcontext()
        with context:
            with autocast_context(device):
                loss_sum = summed_loss(model, batch)
            invalid_loss.logical_or_(~torch.isfinite(loss_sum.detach()))
            local_loss += loss_sum.detach().double()
            scaler.scale(loss_sum * (world_size / global_tokens)).backward()
    scaler.unscale_(optimizer)
    gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm,
                                            error_if_nonfinite=False, foreach=device.type == "cuda")
    flags = torch.stack((invalid_loss, ~torch.isfinite(gradient_norm))).to(torch.int32)
    if distributed():
        dist.all_reduce(flags, op=dist.ReduceOp.MAX)
    bad_loss, bad_gradient = flags.tolist()
    if bad_loss:
        optimizer.zero_grad(set_to_none=True)
        raise FloatingPointError("Non-finite loss; optimizer unchanged. Recover from the committed checkpoint.")
    overflow = bool(bad_gradient)
    if overflow:
        if not scaler.is_enabled():
            raise FloatingPointError("Non-finite FP32 gradients.")
        scaler.update(new_scale=scaler.get_scale() * scaler.get_backoff_factor())
    else:
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
    optimizer.zero_grad(set_to_none=True)
    loss_value = float(reduce_sum(local_loss).item())
    return {"loss_sum": loss_value, "tokens": global_tokens, "loss": loss_value / global_tokens,
            "learning_rate": used_lr, "overflow": overflow,
            "gradient_norm": float(gradient_norm.item())}


''')


[Markdown Cell]
train.py: evaluate


In [75]:
# [Code Cell]
append_module('train.py', r'''@torch.inference_mode()
def evaluate(model: LegalSparseDecoder, cfg: TrainConfig, device: torch.device,
             rank: int, world_size: int) -> dict:
    """Visit every held-out pair exactly once globally, without sampler padding."""
    model.eval()
    dataset = ValidationDataset(cfg)
    subset = Subset(dataset, range(rank, len(dataset), world_size))
    loader = DataLoader(subset, batch_size=cfg.micro_batch_size, shuffle=False,
                        collate_fn=collate_continuations, num_workers=cfg.num_workers,
                        generator=torch.Generator().manual_seed(cfg.seed + 10_000),
                        pin_memory=device.type == "cuda",
                        **({"multiprocessing_context": "spawn", "prefetch_factor": cfg.loader_prefetch_factor,
                            "worker_init_fn": loader_worker_init} if cfg.num_workers else {}))
    totals = torch.zeros(2, dtype=torch.float64, device=device)
    for cpu_batch in tqdm(loader, desc="Held-out continuation NLL", disable=rank != 0, mininterval=1):
        batch = move_batch(cpu_batch, device)
        with autocast_context(device):
            loss = summed_loss(model, batch)
        totals[0] += loss.double()
        totals[1] += (batch["labels"] != -100).sum()
    reduce_sum(totals)
    assert totals[1] > 0 and torch.isfinite(totals).all(), "Invalid validation loss"
    nll = float((totals[0] / totals[1]).item())
    model.train()
    return {"conditional_nll": nll, "conditional_perplexity": math.exp(min(nll, 80)),
            "perplexity_capped": nll > 80, "target_tokens": int(totals[1].item()),
            "examples": len(dataset)}


''')


[Markdown Cell]
train.py: new_cursor


In [76]:
# [Code Cell]
append_module('train.py', r'''def new_cursor() -> dict:
    return {"epoch": 0, "next_batch": 0, "global_step": 0, "optimizer_steps": 0,
            "epoch_loss_sum": 0.0, "epoch_tokens": 0, "consecutive_overflows": 0,
            "epoch_overflows": 0, "best_validation_nll": float("inf"), "history": []}


''')


[Markdown Cell]
train.py: training_loader


In [77]:
# [Code Cell]
append_module('train.py', r'''def training_loader(dataset, cfg: TrainConfig, epoch: int, start_batch: int, rank: int):
    sampler = EpochBatchSampler(len(dataset), cfg.micro_batch_size, rank, cfg.world_size,
                                cfg.seed, epoch, start_batch)
    loader = DataLoader(dataset, batch_sampler=sampler, collate_fn=collate_continuations,
                        num_workers=cfg.num_workers, pin_memory=True, persistent_workers=False,
                        generator=torch.Generator().manual_seed(cfg.seed + epoch * 31 + rank),
                        **({"multiprocessing_context": "spawn", "prefetch_factor": cfg.loader_prefetch_factor, "worker_init_fn": loader_worker_init}
                           if cfg.num_workers else {}))
    return loader, sampler


''')


[Markdown Cell]
train.py: repair_step_log


In [78]:
# [Code Cell]
append_module('train.py', r'''def repair_step_log(path: Path, saved_step: int) -> None:
    """Remove interrupted/replayed log tails newer than the committed cursor."""
    if not path.exists():
        return
    temporary = path.with_name(path.name + ".partial")
    with path.open(encoding="utf-8") as source, temporary.open("w", encoding="utf-8") as target:
        for line in source:
            try:
                row = json.loads(line)
            except ValueError:
                break
            if row["step"] <= saved_step:
                target.write(line)
        target.flush(); os.fsync(target.fileno())
    os.replace(temporary, path)


''')


[Markdown Cell]
train.py: worker


In [79]:
# [Code Cell]
append_module('train.py', r'''def worker(rank: int, cfg: TrainConfig, contract: dict, fingerprint: str,
           port: int, stop_event) -> None:
    """One spawned process per T4; all training collectives follow the same order."""
    os.environ["MASTER_ADDR"], os.environ["MASTER_PORT"] = "127.0.0.1", str(port)
    torch.cuda.set_device(rank)
    device = torch.device("cuda", rank)
    signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
    signal.signal(signal.SIGINT, lambda *_: stop_event.set())
    dist.init_process_group(backend="nccl", rank=rank, world_size=cfg.world_size,
                            timeout=timedelta(minutes=15))
    started = time.monotonic()
    try:
        torch.set_num_threads(1)
        seed_everything(cfg.seed + rank, cfg.deterministic)
        with torch.device(device):
            model = LegalSparseDecoder(cfg)
        assert_parameter_boundary(model)
        ddp = DDP(model, device_ids=[rank], output_device=rank,
                  broadcast_buffers=False, find_unused_parameters=False,
                  gradient_as_bucket_view=True)
        optimizer = build_optimizer(model, cfg)
        dataset = ContinuationDataset(cfg)
        batches_per_epoch = math.ceil(len(dataset) / (cfg.world_size * cfg.micro_batch_size))
        updates_per_epoch = math.ceil(batches_per_epoch / cfg.accumulation_steps)
        scheduler = WarmupCosineScheduler(optimizer, updates_per_epoch * cfg.epochs,
                                         cfg.warmup_updates, cfg.learning_rate, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        manager = CheckpointManager(cfg, fingerprint)
        payload = manager.restore(model, optimizer, scaler, scheduler, device)
        if payload is None:
            parent = load_initial_weights(model, cfg, contract)
            if parent and rank == 0:
                print("Loaded parent weights:", json.dumps(parent), flush=True)
        cursor = new_cursor() if payload is None else payload["cursor"]
        assert 0 <= cursor["epoch"] <= cfg.epochs and 0 <= cursor["next_batch"] <= batches_per_epoch
        assert cursor["next_batch"] % cfg.accumulation_steps == 0 or cursor["next_batch"] == batches_per_epoch
        if payload is not None:
            assert len(payload["rng_by_rank"]) == cfg.world_size
            restore_rng(payload["rng_by_rank"][rank], device)
            del payload
        log_path = Path(cfg.run_dir) / "training_steps.jsonl"
        if rank == 0:
            repair_step_log(log_path, cursor["global_step"])
            print(f"Model D: {model.parameter_count:,} unique trainable parameters", flush=True)
            print(f"Resume: epoch={cursor['epoch']} next_batch={cursor['next_batch']} "
                  f"step={cursor['global_step']} optimizer_step={cursor['optimizer_steps']}", flush=True)
        run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                            cursor, cfg, device, rank, stop_event, started, log_path)
    finally:
        dist.destroy_process_group()


''')


[Markdown Cell]
train.py: run_training_epochs


In [80]:
# [Code Cell]
append_module('train.py', r'''def run_training_epochs(ddp, model, optimizer, scaler, scheduler, manager, dataset,
                        cursor: dict, cfg: TrainConfig, device: torch.device, rank: int,
                        stop_event, started: float, log_path: Path) -> None:
    epoch_bar = tqdm(total=cfg.epochs, initial=cursor["epoch"], desc="Model D epochs", disable=rank != 0)
    stopped = False
    try:
        while cursor["epoch"] < cfg.epochs:
            loader, sampler = training_loader(dataset, cfg, cursor["epoch"], cursor["next_batch"], rank)
            iterator = iter(loader)
            total_updates = math.ceil(sampler.num_batches / cfg.accumulation_steps)
            step_bar = tqdm(total=total_updates,
                            initial=math.ceil(cursor["next_batch"] / cfg.accumulation_steps),
                            desc=f"Epoch {cursor['epoch'] + 1}/{cfg.epochs}", disable=rank != 0,
                            mininterval=1, leave=False)
            try:
                while cursor["next_batch"] < sampler.num_batches:
                    update_started = time.perf_counter()
                    count = min(cfg.accumulation_steps, sampler.num_batches - cursor["next_batch"])
                    batches = [next(iterator) for _ in range(count)]
                    metrics = train_one_update(ddp, optimizer, scaler, scheduler, batches, cfg, device)
                    update_seconds = time.perf_counter() - update_started
                    target_tokens_per_second = metrics["tokens"] / max(update_seconds, 1e-9)
                    cursor["next_batch"] += count
                    cursor["global_step"] += 1
                    cursor["optimizer_steps"] += int(not metrics["overflow"])
                    cursor["epoch_loss_sum"] += metrics["loss_sum"]
                    cursor["epoch_tokens"] += metrics["tokens"]
                    cursor["epoch_overflows"] += int(metrics["overflow"])
                    cursor["consecutive_overflows"] = cursor["consecutive_overflows"] + 1 if metrics["overflow"] else 0
                    assert cursor["optimizer_steps"] == scheduler.steps
                    if rank == 0:
                        with log_path.open("a", encoding="utf-8") as stream:
                            row = {"epoch": cursor["epoch"], "step": cursor["global_step"],
                                   "optimizer_step": cursor["optimizer_steps"],
                                   "loss": metrics["loss"], "learning_rate": metrics["learning_rate"],
                                   "tokens": metrics["tokens"], "overflow": metrics["overflow"],
                                   "update_seconds": update_seconds, "target_tokens_per_second": target_tokens_per_second}
                            stream.write(json.dumps(row, allow_nan=False) + "\n")
                        step_bar.update(1)
                        step_bar.set_postfix(step=cursor["global_step"], loss=f"{metrics['loss']:.4f}",
                                             lr=f"{metrics['learning_rate']:.2e}", tok_s=f"{target_tokens_per_second:.0f}", refresh=False)
                    stopping = any_rank(stop_event.is_set() or time.monotonic() - started >= cfg.max_run_seconds, device)
                    repeated_overflow = cursor["consecutive_overflows"] >= cfg.max_consecutive_overflows
                    if cursor["global_step"] % cfg.checkpoint_every_steps == 0 or stopping or repeated_overflow:
                        manager.save(model, optimizer, scaler, scheduler, cursor, device, rank)
                    if repeated_overflow:
                        raise FloatingPointError("Repeated AMP overflows; checkpoint saved. Inspect the scale/LR before resuming.")
                    if stopping:
                        stopped = True
                        break
            finally:
                step_bar.close()
                del iterator, loader
            if stopped:
                break
            validation = evaluate(model, cfg, device, rank, cfg.world_size)
            improved = validation["conditional_nll"] < cursor["best_validation_nll"]
            if improved:
                cursor["best_validation_nll"] = validation["conditional_nll"]
            record = {"epoch": cursor["epoch"] + 1,
                      "train_nll": cursor["epoch_loss_sum"] / cursor["epoch_tokens"],
                      "training_target_tokens": cursor["epoch_tokens"],
                      "overflows": cursor["epoch_overflows"], **validation}
            cursor["history"].append(record)
            cursor.update(epoch=cursor["epoch"] + 1, next_batch=0, epoch_loss_sum=0.0,
                          epoch_tokens=0, epoch_overflows=0)
            manager.save(model, optimizer, scaler, scheduler, cursor, device, rank, best=improved)
            if rank == 0:
                epoch_bar.update(1); epoch_bar.set_postfix(val_nll=f"{validation['conditional_nll']:.4f}")
                print(json.dumps(record), flush=True)
        if rank == 0:
            atomic_json(Path(cfg.run_dir) / "training_summary.json", {
                "status": "paused_at_saved_update" if stopped else "complete",
                "epoch": cursor["epoch"], "next_batch": cursor["next_batch"],
                "step": cursor["global_step"], "optimizer_steps": cursor["optimizer_steps"],
                "history": cursor["history"], "fingerprint": manager.fingerprint,
            })
    finally:
        epoch_bar.close()


''')


[Markdown Cell]
train.py: require_dual_t4


In [81]:
# [Code Cell]
append_module('train.py', r'''def require_dual_t4() -> None:
    version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
    assert version >= (2, 5), "Use Kaggle's current PyTorch CUDA runtime (>=2.5)."
    assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Select Kaggle accelerator T4 x2."
    assert dist.is_nccl_available(), "The PyTorch build needs NCCL support."
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        assert "T4" in name, f"Expected T4 on device {rank}: {name}"
        print(f"GPU {rank}: {name}, {torch.cuda.get_device_properties(rank).total_memory / 2**30:.1f} GiB")


''')


[Markdown Cell]
train.py: main


In [82]:
# [Code Cell]
append_module('train.py', r'''def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config)); cfg.validate()
    require_dual_t4()
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("TORCH_NCCL_ASYNC_ERROR_HANDLING", "1")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    with exclusive_run_lock(Path(cfg.run_dir)):
        contract = verify_data_contract(cfg)
        report = prepare_indices(cfg, contract)
        fingerprint = runtime_fingerprint(cfg, contract)
        if CheckpointManager(cfg, fingerprint).latest() is None:
            verify_initial_checkpoint(cfg, contract)
        atomic_json(Path(cfg.run_dir) / "training_runtime_identity.json",
                    {"torch": str(torch.__version__), "fingerprint": fingerprint,
                     "experiment_kind": cfg.experiment_kind})
        atomic_json(Path(cfg.run_dir) / "data_report.json", report)
        atomic_json(Path(cfg.run_dir) / "performance_settings.json", {
            "optimization_version": cfg.optimization_version, "fused_adamw": cfg.fused_adamw,
            "loader_workers_per_rank": cfg.num_workers, "prefetch_factor": cfg.loader_prefetch_factor,
            "loss_chunk_tokens": cfg.loss_chunk_tokens, "full_window_coverage": True,
            "epochs": cfg.epochs, "effective_global_batch": cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps})
        with torch.device("meta"):
            count = assert_parameter_boundary(LegalSparseDecoder(cfg))
        print(f"Verified Model D parameter count: {count:,}", flush=True)
        # Uncommitted pipeline-owned temporaries cannot be restored.
        checkpoint_dir = Path(cfg.run_dir) / "checkpoints"
        if checkpoint_dir.exists():
            for name in ("slot_0.pt.partial", "slot_1.pt.partial", "best.pt.partial",
                         "slot_0.pt.json.partial", "slot_1.pt.json.partial", "best.pt.json.partial"):
                (checkpoint_dir / name).unlink(missing_ok=True)
        context = mp.get_context("spawn")
        stop_event = context.Event()
        signal.signal(signal.SIGTERM, lambda *_: stop_event.set())
        signal.signal(signal.SIGINT, lambda *_: stop_event.set())
        with socket.socket() as sock:
            sock.bind(("127.0.0.1", 0)); port = sock.getsockname()[1]
        processes = mp.spawn(worker, args=(cfg, contract, fingerprint, port, stop_event),
                             nprocs=cfg.world_size, join=False)
        while not processes.join(timeout=1):
            pass


# CELL: Bounded loader processes and recorded execution settings
''')


[Markdown Cell]
train.py: loader_worker_init


In [83]:
# [Code Cell]
append_module('train.py', r'''def loader_worker_init(worker_id: int) -> None:
    """One data worker per GPU; prevent nested CPU/tokenizer thread oversubscription."""
    torch.set_num_threads(1)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

if __name__ == "__main__":
    main()
''')


[Markdown Cell]
For $R$ ranks, scale each microbatch's summed loss by $R/N_{\rm global}$;
DDP's gradient average gives the global token-mean gradient, including variable
lengths and short final accumulation groups. `no_sync()` encloses forward and
backward for nonfinal microbatches. Unscale before clipping. Any rank's nonfinite
gradient makes all ranks skip the optimizer and back off their scales together.
[DDP](https://docs.pytorch.org/docs/stable/notes/ddp),
[AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples).


[Markdown Cell]
Each held-out example is visited exactly once globally. Evaluate the raw module so unequal rank batch counts do not trigger DDP forward collectives; then reduce summed NLL and target tokens. Best selection uses token-weighted conditional NLL. Perplexity is specific to this conditional objective; the provided evaluator measures BERTScore/ROUGE/BLEU separately.


[Markdown Cell]
Record the next rank-local microbatch, completed accumulation groups, successful optimizer updates, and epoch totals. Resume jumps directly to that batch; remove replayed log tails after a corrupt-checkpoint fallback. Sampling and item contents are deterministic.


[Markdown Cell]
One `spawn` process owns each T4, with `backend='nccl'`. Restore full training state and that rank's RNG before loading the next saved batch. Rank 0 alone shows tqdm progress with epoch, step, loss, and LR. The production code uses DDP throughout.


[Markdown Cell]
Automatically save every **500 completed accumulation groups** and **at every epoch end**. SIGINT/SIGTERM or the 10.5-hour limit requests a save at the next update boundary. An abrupt kill can lose up to 499 groups after the last checkpoint. Resume is exact from the saved cursor; working files must be preserved across sessions.


[Markdown Cell]
Require exactly two T4s and an NCCL-enabled PyTorch >=2.5 build. Verify inputs and build indices in the parent, then launch `torch.multiprocessing.spawn` from the emitted module. No production fallback backend silently changes training. Set `resume_dir` to attached checkpoint files and receipts when working storage did not survive.


[Markdown Cell]
## 12. Continuation generation
Greedy decoding suppresses control tokens and uses the sparse decoder's rotary K/V cache. Generate an extra word to close the 128th word, then export exactly 128 words without the prompt. Token/context exhaustion fails explicitly and preserves completed predictions; no filler is added.


[Markdown Cell]
infer.py: imports and module constants


In [84]:
# [Code Cell]
append_module('infer.py', r'''import argparse
from pathlib import Path
import hashlib
import json
import os
import shutil
import sqlite3
import zipfile
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .config import TrainConfig, SOURCES, SPECIAL_TOKENS, resolve_training_config
from .data import read_json, sha256_file, verify_data_contract
from .model import LegalSparseDecoder, decode_one_token
from .state import CheckpointManager, atomic_json, runtime_fingerprint, seed_everything, exclusive_run_lock
from .train import autocast_context


''')


[Markdown Cell]
infer.py: generate_continuation


In [85]:
# [Code Cell]
append_module('infer.py', r'''@torch.inference_mode()
def generate_continuation(model: LegalSparseDecoder, tokenizer: Tokenizer, prompt: str,
                          device: torch.device, token_limit: int, target_words: int = 128) -> str:
    """Generate until an extra word closes the requested final word; return 128.

    Token count is not word count. EOS/control tokens are suppressed; exhausting
    the context/token bound fails explicitly instead of inventing padding text.
    """
    model.eval()
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    ids = [1] + prompt_ids[-(model.cfg.max_prompt_tokens - 1):]
    token_ids = torch.tensor([ids], device=device)
    generated = []
    limit = min(token_limit, model.cfg.max_seq_tokens - len(ids) + 1)
    forbidden = [tokenizer.token_to_id(special) for special in SPECIAL_TOKENS]
    with autocast_context(device):
        logits, cache = model.prefill(token_ids)
    for step in range(limit):
        if step:
            with autocast_context(device):
                logits = decode_one_token(model, token, cache)
        logits = logits.float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite generation logits")
        logits[:, forbidden] = float("-inf")
        chosen = int(logits.argmax(dim=-1).item())
        generated.append(chosen)
        token = torch.tensor([[chosen]], device=device)
        if (step + 1) % 8 == 0 or step + 1 == limit:
            text = tokenizer.decode(generated, skip_special_tokens=True)
            words = text.split()
            if len(words) >= target_words + 1:
                continuation = " ".join(words[:target_words])
                assert len(continuation.split()) == target_words
                return continuation
    raise RuntimeError(f"Could not close {target_words} words within {limit} generated tokens. "
                       "Inspect training/generation quality or use a larger causal context capacity in a new experiment.")


''')


[Markdown Cell]
infer.py: load_inference_model


In [86]:
# [Code Cell]
append_module('infer.py', r'''def load_inference_model(cfg: TrainConfig, fingerprint: str, device: torch.device):
    manager = CheckpointManager(cfg, fingerprint)
    latest = manager.latest()
    if latest is None:
        raise FileNotFoundError("No trained checkpoint exists. Run distributed training first.")
    chosen = latest
    roots = [manager.directory] + ([Path(cfg.resume_dir)] if cfg.resume_dir else [])
    best_candidates = []
    for root in roots:
        path, receipt_path = root / "best.pt", root / "best.pt.json"
        if path.exists() and receipt_path.exists():
            try:
                receipt = read_json(receipt_path)
                assert receipt["fingerprint"] == fingerprint and path.stat().st_size == receipt["bytes"]
                assert sha256_file(path) == receipt["sha256"]
                score = receipt.get("validation_nll")
                assert score is None or isinstance(score, (int, float))
                best_candidates.append((path, receipt))
            except (OSError, ValueError, TypeError, KeyError, AssertionError):
                continue
    if best_candidates:
        chosen = min(best_candidates, key=lambda item: (
            item[1].get("validation_nll") if item[1].get("validation_nll") is not None else float("inf"),
            -item[1]["step"], -item[1]["generation"]))
    payload = torch.load(chosen[0], map_location="cpu", weights_only=True)
    assert payload["fingerprint"] == fingerprint
    model = LegalSparseDecoder(cfg)
    model.load_state_dict(payload["model"], strict=True)
    model.to(device).eval()
    receipt = chosen[1]
    del payload
    return model, receipt


''')


[Markdown Cell]
infer.py: generation_records


In [87]:
# [Code Cell]
append_module('infer.py', r'''def generation_records(cfg: TrainConfig, kind: str, input_json: str | None):
    """Use exact official string IDs; development records come only from held-out data."""
    if kind == "test":
        if input_json is None:
            raise ValueError("Provide --input-json pointing to the official Phase-1 test prompts.")
        payload = read_json(Path(input_json))
        assert isinstance(payload, dict) and payload
        for identifier, row in payload.items():
            assert isinstance(identifier, str) and isinstance(row, dict)
            assert isinstance(row.get("text"), str) and len(row["text"].split()) == 512
            yield {"id": identifier, "prompt": row["text"]}
    else:
        counts = dict.fromkeys(SOURCES, 0)
        with (Path(cfg.data_dir) / "validation_pairs.jsonl").open(encoding="utf-8") as stream:
            for line in stream:
                row = json.loads(line)
                source = row["source"]
                if counts[source] < cfg.generation_examples_per_source:
                    counts[source] += 1
                    yield row
        assert all(count == cfg.generation_examples_per_source for count in counts.values()), counts


''')


[Markdown Cell]
infer.py: export_predictions


In [88]:
# [Code Cell]
append_module('infer.py', r'''def export_predictions(path: Path, predictions: dict) -> None:
    assert predictions and all(len(row["text"].split()) == 128 for row in predictions.values())
    atomic_json(path, predictions)


''')


[Markdown Cell]
infer.py: run_generation


In [89]:
# [Code Cell]
append_module('infer.py', r'''def run_generation(cfg: TrainConfig, kind: str, input_json: str | None = None,
                   device: torch.device | None = None) -> None:
    cfg.validate()
    if device is None:
        assert torch.cuda.is_available(), "Generation requires the Kaggle CUDA runtime."
        device = torch.device("cuda", 0)
    seed_everything(cfg.seed, cfg.deterministic)
    contract = verify_data_contract(cfg)
    fingerprint = runtime_fingerprint(cfg, contract)
    model, receipt = load_inference_model(cfg, fingerprint, device)
    tokenizer = Tokenizer.from_file(str(Path(cfg.data_dir) / "tokenizer.json"))
    directory = Path(cfg.run_dir) / "generation" / kind
    directory.mkdir(parents=True, exist_ok=True)
    records = list(generation_records(cfg, kind, input_json))
    assert len({row["id"] for row in records}) == len(records)
    prompt_truncations = sum(len(tokenizer.encode(row["prompt"], add_special_tokens=False).ids)
                             > cfg.max_prompt_tokens - 1 for row in records)
    namespace = hashlib.sha256(json.dumps({"checkpoint": receipt["sha256"], "kind": kind,
                "tokens": cfg.generation_token_limit, "words": 128}, sort_keys=True).encode()).hexdigest()
    connection = sqlite3.connect(directory / "generation_cache.sqlite")
    connection.execute("PRAGMA journal_mode=WAL")
    connection.execute("PRAGMA synchronous=FULL")
    connection.execute("CREATE TABLE IF NOT EXISTS predictions (namespace TEXT, id TEXT, "
                       "prompt_sha TEXT, text TEXT, PRIMARY KEY(namespace,id))")
    predictions, truth = {}, {}
    try:
        for row in tqdm(records, desc=f"Model D {kind}: 128-word continuations"):
            prompt_sha = hashlib.sha256(row["prompt"].encode()).hexdigest()
            cached = connection.execute("SELECT prompt_sha,text FROM predictions WHERE namespace=? AND id=?",
                                        (namespace, row["id"])).fetchone()
            if cached is not None and cached[0] == prompt_sha and len(cached[1].split()) == 128:
                text = cached[1]
            else:
                if shutil.disk_usage(directory).free < cfg.min_free_bytes + (1 << 20):
                    raise OSError("Insufficient disk space for durable generation outputs.")
                text = generate_continuation(model, tokenizer, row["prompt"], device, cfg.generation_token_limit)
                connection.execute("INSERT OR REPLACE INTO predictions VALUES (?,?,?,?)",
                                   (namespace, row["id"], prompt_sha, text))
                connection.commit()  # Each completed example survives a later failure.
            predictions[row["id"]] = {"text": text}
            if "reference" in row:
                truth[row["id"]] = {"text": row["reference"]}
        prediction_path = directory / "prediction.json"
        export_predictions(prediction_path, predictions)
        if truth:
            assert truth.keys() == predictions.keys()
            atomic_json(directory / "truth.json", truth)
        if kind == "test":
            temporary = directory / "submission_model_d.zip.partial"
            with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
                archive.write(prediction_path, arcname="prediction.json")
            with temporary.open("r+b") as stream:
                os.fsync(stream.fileno())
            os.replace(temporary, directory / "submission_model_d.zip")
        atomic_json(directory / "generation_report.json", {
            "checkpoint_sha256": receipt["sha256"], "checkpoint_epoch": receipt["epoch"],
            "checkpoint_step": receipt["step"], "examples": len(predictions),
            "words_per_continuation": 128, "decoding": "greedy with cached causal local/dilated/landmark rotary K/V",
            "prompt_tail_limit": cfg.max_prompt_tokens - 1,
            "prompt_truncations": prompt_truncations,
            "token_limit": cfg.generation_token_limit, "data_fingerprint": contract["fingerprint"],
        })
        print(f"Saved {len(predictions)} continuations to {prediction_path}", flush=True)
    finally:
        connection.close()


''')


[Markdown Cell]
infer.py: main


In [90]:
# [Code Cell]
append_module('infer.py', r'''def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--kind", choices=("validation", "test"), default="validation")
    parser.add_argument("--input-json")
    args = parser.parse_args()
    cfg = resolve_training_config(TrainConfig.load(args.config))
    with exclusive_run_lock(Path(cfg.run_dir)):
        run_generation(cfg, args.kind, args.input_json)


if __name__ == "__main__":
    main()
''')


[Markdown Cell]
Use the full best held-out checkpoint when available; otherwise use the latest committed checkpoint. Verify its receipts and compatibility before loading weights. The tokenizer remains identical to training.


[Markdown Cell]
Generate two held-out examples per legal source by default. Optional official inference reads the attached prompt JSON and preserves exact string IDs. Output records contain only `text` with the continuation, matching the supplied evaluator.


[Markdown Cell]
Commit each completed prediction to SQLite. Resume skips entries only when checkpoint, prompt, and decoding settings match. Official inference creates `prediction.json` and a ZIP containing it at the archive root. No external submission is performed.


[Markdown Cell]
## 13. Independent sparse attention oracle
Only this short verification routine constructs a dense mask. It independently
expresses local, dilated, and global conditions over all query/key pairs, then
compares the gathered attention's outputs **and input/projection gradients** with
dense masked SDPA. A 37-position synthetic example crosses window/dilation/global
boundaries, includes overlapping routes, right padding, multiple query chunks,
and checkpointed backward. The production model's parameter guard stays active;
these small standalone attention modules are solely a mathematical check.


[Markdown Cell]
sanity.py: imports and module constants


In [91]:
# [Code Cell]
append_module('sanity.py', r'''from dataclasses import replace
from copy import deepcopy
import gc
import torch
from torch.nn import functional as F
from .config import TrainConfig, assert_parameter_boundary
from .model import (LegalSparseDecoder, RotaryEmbedding, SparseCausalSelfAttention,
                    sparse_key_indices, decode_one_token)
from .data import collate_continuations


''')


[Markdown Cell]
sanity.py: sparse_attention_sanity


In [92]:
# [Code Cell]
append_module('sanity.py', r'''def sparse_attention_sanity(device: torch.device) -> dict:
    """Compare sparse outputs AND gradients to an independent dense mathematical oracle.

    Only this short verification function constructs a dense mask. Production
    attention uses gathered keys. Tiny attention modules add no runtime model.
    """
    cfg = replace(TrainConfig(), d_model=32, n_heads=4, dropout=0.0,
                  local_window=4, dilation_base=3, dilation_cycle=2,
                  dilated_links=5, global_stride=7, attention_query_chunk=5)
    sparse = SparseCausalSelfAttention(cfg, 1).to(device).train()
    dense = deepcopy(sparse)
    ids = torch.arange(37, device=device)
    valid = torch.ones((2, 37), dtype=torch.bool, device=device)
    valid[1, 31:] = False
    gap = ids[:, None] - ids[None, :]
    local = (gap >= 0) & (gap < cfg.local_window)
    distance = gap - cfg.local_window
    dilated = (distance >= 0) & (distance.remainder(dense.dilation) == 0) & (
        distance < cfg.dilated_links * dense.dilation)
    global_keys = ids[None, :].remainder(cfg.global_stride) == 0
    pattern = (gap >= 0) & (local | dilated | global_keys)
    indices, allowed = sparse_key_indices(ids, 37, cfg.local_window, sparse.dilation,
                                         cfg.dilated_links, cfg.global_stride)
    for position in range(37):
        chosen = indices[position, allowed[position]].tolist()
        assert chosen == torch.where(pattern[position])[0].tolist()
        assert len(chosen) == len(set(chosen)) and max(chosen) <= position
    # Window/dilation/global overlaps occur here and must not bias the softmax.
    x1 = torch.randn(2, 37, cfg.d_model, device=device, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    q, k, v = dense.project(x2, ids)
    oracle = F.scaled_dot_product_attention(q, k, v,
        attn_mask=pattern[None, None] & valid[:, None, None, :], dropout_p=0.0)
    oracle = dense.out(oracle.transpose(1, 2).contiguous().flatten(2))
    gathered = sparse(x1, valid)
    torch.testing.assert_close(gathered, oracle, rtol=2e-4, atol=2e-6)
    upstream = torch.randn_like(oracle)
    (gathered * upstream).sum().backward()
    (oracle * upstream).sum().backward()
    gradient_error = 0.0
    for left, right in zip((x1, *sparse.parameters()), (x2, *dense.parameters())):
        torch.testing.assert_close(left.grad, right.grad, rtol=4e-4, atol=5e-6)
        gradient_error = max(gradient_error, float((left.grad - right.grad).abs().max()))
    # Default full-context fanout: fewer keys than a 2048-key dense query.
    default = TrainConfig()
    count = default.local_window + default.dilated_links + (
        default.max_seq_tokens + default.global_stride - 1) // default.global_stride
    assert count == 152
    return {"dense_sparse_output_max_abs_error": float((gathered - oracle).abs().max()),
            "dense_sparse_gradient_max_abs_error": gradient_error,
            "causal_routes_and_deduplication": "passed", "right_padding": "passed",
            "default_max_candidate_keys": count, "query_chunk_checkpointed_backward": "passed"}


''')


[Markdown Cell]
sanity.py: architecture_sanity


In [93]:
# [Code Cell]
append_module('sanity.py', r'''def architecture_sanity(cfg: TrainConfig, device: torch.device | None = None) -> dict:
    """Exercise the full production architecture on short sequences before DDP."""
    device = torch.device("cpu") if device is None else device
    threads = torch.get_num_threads()
    torch.set_num_threads(min(4, threads))
    try:
        sparse_report = sparse_attention_sanity(device)
        with torch.device("meta"):
            count = assert_parameter_boundary(LegalSparseDecoder(cfg))
        expected = cfg.vocab_size * cfg.d_model + cfg.n_layers * (
            4 * cfg.d_model ** 2 + 3 * cfg.d_model * cfg.d_ff + 2 * cfg.d_model) + cfg.d_model
        assert count == expected
        with torch.device(device):
            model = LegalSparseDecoder(cfg).eval()
        ids = torch.randint(6, cfg.vocab_size, (2, 12), device=device)
        mask = torch.ones_like(ids, dtype=torch.bool)
        mask[1, -3:] = False
        with torch.inference_mode():
            logits = model(ids, mask)
            assert logits.shape == (2, 12, cfg.vocab_size) and torch.isfinite(logits).all()
            changed = ids.clone(); changed[:, 6:] = (changed[:, 6:] + 11) % cfg.vocab_size
            torch.testing.assert_close(logits[:, :6], model(changed, mask)[:, :6], rtol=1e-4, atol=3e-6)
            padded = ids.clone(); padded[1, -3:] = 9
            torch.testing.assert_close(logits[1, :9], model(padded, mask)[1, :9], rtol=1e-4, atol=3e-6)
            # Exercise both the real 128-token window and the landmark at 256.
            length = min(cfg.max_seq_tokens, max(cfg.local_window + 9, cfg.global_stride + 17))
            unpadded = torch.randint(6, cfg.vocab_size, (1, length), device=device)
            full = model(unpadded)
            prefix_length = max(1, length - 20)
            first, cache = model.prefill(unpadded[:, :prefix_length])
            cached = [first]
            for position in range(prefix_length, unpadded.shape[1]):
                cached.append(decode_one_token(model, unpadded[:, position:position + 1], cache))
            incremental = torch.stack(cached, dim=1)
            torch.testing.assert_close(full[:, prefix_length - 1:], incremental, rtol=2e-4, atol=1e-5)
            cache_error = float((full[:, prefix_length - 1:] - incremental).abs().max())
            assert cache.position == length
            assert all(k.shape[2] == length and v.shape == k.shape for k, v in cache.key_values)
            labels = torch.randint(6, cfg.vocab_size, ids.shape, device=device)
            labels[1, -3:] = -100
            labels[:, :2] = -100
            dense_loss = F.cross_entropy(logits.flatten(0, 1).float(), labels.flatten(),
                                         ignore_index=-100, reduction="sum")
            chunk_loss = model(ids, mask, labels)
            torch.testing.assert_close(dense_loss, chunk_loss, rtol=1e-5, atol=1e-4)
            rotary = RotaryEmbedding(8, cfg.rope_base)
            q, k = torch.randn(1, 2, 4, 8, device=device), torch.randn(1, 2, 4, 8, device=device)
            positions = torch.arange(4, device=device)
            q1, k1 = rotary(q, k, positions)
            q2, k2 = rotary(q, k, positions + 17)
            torch.testing.assert_close(q1 @ k1.transpose(-1, -2), q2 @ k2.transpose(-1, -2),
                                       rtol=1e-4, atol=4e-6)
        batch = collate_continuations([
            {"input_ids": [1, 6, 7], "labels": [6, 7, 2]},
            {"input_ids": [1, 8], "labels": [-100, 9]},
        ])
        assert batch["labels"][1].tolist() == [-100, 9, -100]
        assert batch["attention_mask"][1].tolist() == [True, True, False]
        model.train()
        batch = {name: value.to(device) for name, value in batch.items()}
        loss = model(batch["input_ids"], batch["attention_mask"], batch["labels"])
        loss.backward()
        for parameter in (model.token_embedding.weight, model.layers[0].attention.qkv.weight,
                          model.layers[-1].ffn.gate.weight):
            assert parameter.grad is not None and torch.isfinite(parameter.grad).all()
            assert parameter.grad.abs().sum() > 0
        result = {"parameters": count, "logit_shape": list(logits.shape), "device": str(device),
                  "sparse_attention": sparse_report, "cache_test_length": length,
                  "causal_isolation": "passed", "right_padding": "passed", "rotary_relative_positions": "passed",
                  "cached_logits_max_abs_error": cache_error, "dense_vs_chunk_loss": "passed",
                  "activation_checkpointed_gradient_flow": "passed"}
        del model, loss, logits, full, incremental, cache, chunk_loss, dense_loss
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return result
    finally:
        torch.set_num_threads(threads)
''')


[Markdown Cell]
Check the full **110,119,680-parameter** production model: causal isolation,
padding isolation, relative rotary positions, sparse/full cached logits, exact
vocabulary loss, and gradients through attention, FFN, and tied vocabulary.
The default cache test consumes 273 positions and crosses the landmark at 256.
Use GPU 0 briefly when available, then release memory before spawning training.


[Markdown Cell]
## 14. Materialize the independent runtime
Write only this notebook's code under its run directory. Save the resolved configuration before launching; child processes read this JSON. Companion workspace files are not required on Kaggle.


In [94]:
# [Code Cell]
code_dir = Path(cfg.run_dir) / "code"
package_dir = code_dir / "model_d"
package_dir.mkdir(parents=True, exist_ok=True)
module_parts["__init__.py"] = ['"""From-scratch sparse causal legal LM and resumable DDP runtime."""\n']
for filename, sections in module_parts.items():
    source = "".join(sections)
    compile(source, filename, "exec")
    path = package_dir / filename
    temporary = path.with_suffix(".partial")
    with temporary.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(source); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
config_path = Path(cfg.run_dir) / "train_config.json"
cfg.save(config_path)
sys.path.insert(0, str(code_dir))
for module_name in list(sys.modules):
    if module_name == "model_d" or module_name.startswith("model_d."):
        del sys.modules[module_name]
importlib.invalidate_caches()
print("Importable runtime:", package_dir)
print("Configuration:", config_path)


Importable runtime: /kaggle/working/model_d_fast_run/code/model_d
Configuration: /kaggle/working/model_d_fast_run/train_config.json


In [95]:
# [Code Cell]
from model_d.sanity import architecture_sanity
sanity_device = torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")
sanity_report = architecture_sanity(cfg, sanity_device)
print(json.dumps(sanity_report, indent=2))
assert 50_000_000 < sanity_report["parameters"] < 300_000_000

import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


/kaggle/working/model_d_fast_run/code/model_d/sanity.py:60: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return {"dense_sparse_output_max_abs_error": float((gathered - oracle).abs().max()),


{
  "parameters": 110119680,
  "logit_shape": [
    2,
    12,
    32768
  ],
  "device": "cuda:0",
  "sparse_attention": {
    "dense_sparse_output_max_abs_error": 5.960464477539063e-08,
    "dense_sparse_gradient_max_abs_error": 1.1920928955078125e-06,
    "causal_routes_and_deduplication": "passed",
    "right_padding": "passed",
    "default_max_candidate_keys": 152,
    "query_chunk_checkpointed_backward": "passed"
  },
  "cache_test_length": 273,
  "causal_isolation": "passed",
  "right_padding": "passed",
  "rotary_relative_positions": "passed",
  "cached_logits_max_abs_error": 4.291534423828125e-06,
  "dense_vs_chunk_loss": "passed",
  "activation_checkpointed_gradient_flow": "passed"
}


[Markdown Cell]
## 15. Launch and resume training
The parent verifies input checksums, indexes document windows, and starts two NCCL workers. tqdm output streams live. Re-running resumes the latest intact compatible checkpoint. Interrupt once to request a checkpoint, and leave this cell running until the trainer reports its saved boundary.


In [96]:
# [Code Cell]
def run_streamed(command: list[str], working_directory: Path) -> None:
    """Forward subprocess output, including tqdm carriage returns, in Jupyter."""
    import select
    import codecs
    process = subprocess.Popen(command, cwd=str(working_directory), stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=0, env=os.environ.copy())
    interrupted = False
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    assert process.stdout is not None
    try:
        while True:
            try:
                ready, _, _ = select.select([process.stdout], [], [], 1.0)
                if ready:
                    chunk = os.read(process.stdout.fileno(), 8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                interrupted = True
                print("\nRequesting a checkpoint at the next update boundary; keep this cell running.")
                process.send_signal(signal.SIGINT)
        return_code = process.wait()
        sys.stdout.write(decoder.decode(b"", final=True)); sys.stdout.flush()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
        if interrupted:
            print("Trainer exited after the checkpoint request.")
    finally:
        process.stdout.close()

run_streamed([sys.executable, "-u", "-m", "model_d.train", "--config", str(config_path)], code_dir)


Preprocessed data: /kaggle/input/datasets/azadbharti/dataset-tokenized-smallest-25m
GPU 0: Tesla T4, 14.6 GiB
GPU 1: Tesla T4, 14.6 GiB
Verify preprocessing receipts: 100%|██████████| 6/6 [00:00<00:00,  9.16it/s]
Verified Model D parameter count: 110,119,680
Model D: 110,119,680 unique trainable parameters
Resume: epoch=0 next_batch=0 step=0 optimizer_step=0
Epoch 1/1: 100%|██████████| 1828/1828 [5:33:13<00:00,  8.78s/it, loss=2.5395, lr=3.00e-05, step=1827, tok_s=935] 
                                                                                                                
Model D epochs: 100%|██████████| 1/1 [5:34:47<00:00, 20087.79s/it, val_nll=3.4150]


In [97]:
# [Code Cell]
for report_name in ("training_summary.json", "data_report.json"):
    report_path = Path(cfg.run_dir) / report_name
    if report_path.exists():
        print(report_name)
        print(report_path.read_text(encoding="utf-8"))


training_summary.json
{
  "status": "complete",
  "epoch": 1,
  "next_batch": 0,
  "step": 1828,
  "optimizer_steps": 1828,
  "history": [
    {
      "epoch": 1,
      "train_nll": 3.9001000199014784,
      "training_target_tokens": 24981275,
      "overflows": 0,
      "conditional_nll": 3.4150315478939044,
      "conditional_perplexity": 30.417909033395322,
      "perplexity_capped": false,
      "target_tokens": 99232,
      "examples": 632
    }
  ],
  "fingerprint": "b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352"
}

data_report.json
{
  "train_examples": 29242,
  "cached_source_tokens": {
    "supreme_court": 8750000,
    "high_courts": 8750000,
    "acts": 3750000,
    "tribunals": 3750000
  },
  "eligible_target_tokens": {
    "supreme_court": 8748879,
    "high_courts": 8733603,
    "acts": 3749528,
    "tribunals": 3749265
  },
  "one_token_documents_dropped": {
    "supreme_court": 0,
    "high_courts": 0,
    "acts": 0,
    "tribunals": 0
  },
  "global_b

[Markdown Cell]
## 16. Generate balanced development continuations
The GPU training workers have exited. This separate one-GPU inference process writes matching prediction/truth JSON maps for the supplied local evaluator. Completed examples survive a later generation failure.


In [98]:
# [Code Cell]
print("Saved training output:", cfg.run_dir)
print("Use 05_Evaluate_Any_Model.ipynb to measure BERTScore and generate verified ZIPs.")


Saved training output: /kaggle/working/model_d_fast_run
Use 05_Evaluate_Any_Model.ipynb to measure BERTScore and generate verified ZIPs.


[Markdown Cell]
## 17. Optional official inference
Set `OFFICIAL_TEST_JSON` in the configuration cell to the attached official prompt file. IDs are preserved exactly. Inspect the generated JSON and ZIP before submitting them yourself.


In [99]:
# [Code Cell]
if OFFICIAL_TEST_JSON is not None:
    test_path = Path(OFFICIAL_TEST_JSON)
    assert test_path.is_file(), test_path
    run_streamed([sys.executable, "-u", "-m", "model_d.infer", "--config", str(config_path),
                  "--kind", "test", "--input-json", str(test_path)], code_dir)
    print("Submission ZIP:", Path(cfg.run_dir) / "generation/test/submission_model_d.zip")
else:
    print("Set OFFICIAL_TEST_JSON to enable official inference.")


Set OFFICIAL_TEST_JSON to enable official inference.


[Markdown Cell]
## 18. Preserve the run and interpret results

Save `model_d_run/`, including configuration, code, logs, and checkpoint files
with their SHA-256 receipts. For a fresh session, attach saved checkpoints and
set `resume_dir` to their folder. Keep architecture, data, learning settings,
runtime source, and PyTorch version consistent. A discarded working directory
cannot be recovered automatically.

Checkpoint storage is capped at 6GiB with a 1GiB free-space reserve. An additional
snapshot must fit while replacing a slot. The token corpus stays in read-only
input; old per-epoch/per-step checkpoints do not accumulate. If GPU memory is
insufficient, change batch/context settings in a fresh run directory so resume
does not silently change the experiment.

Training NLL averages all document transitions; validation NLL averages only
reference tokens conditioned on a held-out prompt. Use the supplied evaluator
for BERTScore F1, ROUGE-L, and BLEU. Record BPE crops, context length, supervised
tokens, skipped AMP updates, and generation settings in your report. Greedy
generation is a baseline to evaluate, not a claim about legal-text quality.

Local QA exercises actual-architecture GPU shapes/backward and small two-rank
CPU Gloo recovery cases. Production remains two-rank NCCL. A local single GPU
does not verify Kaggle T4 throughput or two-device FP16/NCCL behavior.
This is the final notebook in the planned Phase-1 sequence.
